## Pythonizing Vincent!

In [1]:
# ============================================================
# CELL 1: Vincent Pπ construction and DSD sampling functions
# ============================================================
# This cell defines the core functions for Vincent's determinantal
# sampling design.
#
# Main functions:
#   1. Ppi(Pi)
#      Builds an orthonormal matrix V from the first-order inclusion
#      probabilities Pi. The projection kernel is K = V @ V.T, and
#      its diagonal should satisfy diag(K) ≈ Pi.
#
#   2. Drawing_Dsd(v)
#      Draws samples from the fixed-size determinantal sampling design
#      using the matrix V.
#
# After using Ppi(Pi), we should check:
#   - diag(V @ V.T) ≈ Pi
#   - V.T @ V ≈ I
#   - (V @ V.T) @ (V @ V.T) ≈ V @ V.T
# ============================================================



import numpy as np
from typing import Union, Optional


# ------------------------------------------------------------
# 1) Ppi: build the orthogonal matrix V from first-order Pis
# ------------------------------------------------------------

def Ppi(Pi):
    """
    Robust Python version of Vincent's Pπ construction.

    Pi must satisfy:
        0 < Pi_i < 1
        sum(Pi) is an integer sample size n

    Returns:
        V such that K = V @ V.T is approximately a projection matrix
        with diag(K) ≈ Pi.
    """
    Pi = np.asarray(Pi, dtype=float).ravel()
    N = Pi.size

    if N < 2:
        raise ValueError("Pi must have length greater than 1.")

    if np.any(Pi <= 0) or np.any(Pi >= 1):
        raise ValueError("Pi must satisfy 0 < Pi_i < 1.")

    sum_pi = float(Pi.sum())
    n_int = int(round(sum_pi))

    if not np.isclose(sum_pi, n_int, atol=1e-8, rtol=1e-8):
        raise ValueError(
            f"sum(Pi) must be an integer. Got sum(Pi)={sum_pi:.12f}."
        )

    if n_int <= 0:
        raise ValueError("sum(Pi) must be at least 1.")

    s_vals = np.zeros(N, dtype=float)
    c_vals = np.zeros(N, dtype=float)
    alpha = np.zeros(N, dtype=float)

    kr = [None] * n_int

    cum_sum = 0.0
    r = 1
    r_prev = 0
    tol = 1e-10

    for k in range(N):
        prev_sum = cum_sum
        cum_sum += Pi[k]

        # Robust crossing check.
        # The tolerance avoids missing the last crossing because of floating error.
        if cum_sum >= r - tol:
            if r <= n_int:
                alpha_k = r - prev_sum

                # Numerical safety: alpha should be between 0 and Pi[k].
                alpha_k = min(max(alpha_k, 0.0), Pi[k])

                alpha[k] = alpha_k
                kr[r - 1] = k

                denom = 1.0 - alpha[k]
                if denom <= 0:
                    denom = tol

                val = np.sqrt((1.0 - Pi[k]) / denom)
                s_vals[k] = np.clip(val, 0.0, 1.0)

                r_prev = r
                r += 1
        else:
            denom = r_prev + 1 - prev_sum
            if denom <= 0:
                denom = tol

            val = np.sqrt(Pi[k] / denom)
            s_vals[k] = np.clip(val, 0.0, 1.0)

        c_vals[k] = np.sqrt(max(0.0, 1.0 - s_vals[k] ** 2))

    # If only the final crossing is missed because of numerical precision,
    # assign it to the last unit. Otherwise stop.
    missing = [i for i, x in enumerate(kr) if x is None]

    if missing:
        if missing == [n_int - 1] and np.isclose(cum_sum, n_int, atol=1e-8, rtol=1e-8):
            kr[-1] = N - 1
        else:
            raise RuntimeError(
                f"Ppi construction failed: missing crossings {missing}. "
                f"sum(Pi)={cum_sum:.12f}, n={n_int}."
            )

    V = np.zeros((N, n_int), dtype=float)
    V[0, 0] = 1.0

    for r_idx in range(1, n_int):
        kpos = kr[r_idx - 1]
        if kpos + 1 < N:
            V[kpos + 1, r_idx] = 1.0
        else:
            raise RuntimeError("Ppi construction failed: invalid crossing position.")

    for k in range(N - 1):
        L = V[k, :].copy()
        M = V[k + 1, :].copy()

        V[k, :] = s_vals[k] * L - c_vals[k] * M
        V[k + 1, :] = c_vals[k] * L + s_vals[k] * M

    return V

# ------------------------------------------------------------
# 2) DSD sampling (real and complex versions)
# ------------------------------------------------------------

def Drawing_Dsd(
    v: Union[np.ndarray, list],
    s: int = 1,
    B: bool = False,
    seed: Optional[int] = None,
):
    """
    Python/Numpy port of the R function Drawing_Dsd(v, s=1, B=FALSE, seed=NULL).

    Parameters
    ----------
    v : array-like, shape (N, n)
        Matrix of (real or complex) vectors used in DSD.
    s : int, default 1
        Number of samples (replicates).
    B : bool, default False
        If True: return 0/1 indicator vector(s).
        If False: return indices of selected units (1-based, to match R).
    seed : int or None
        Seed for reproducibility.

    Returns
    -------
    If s == 1:
        1D array of length N (if B=True) or selected indices (if B=False).
    If s > 1:
        2D array of shape (N, s) (if B=True) or (n, s) with indices per sample.
    """
    v = np.asarray(v)
    rng = np.random.default_rng(seed)

    if np.iscomplexobj(v):
        return _dsd_sampling_mult_complex(v, s, B, rng)
    else:
        return _dsd_sampling_mult(v, s, B, rng)


# ---------------------- helpers: real case ---------------------- #

def _dsd_sampling_mult(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]  # treat as (N,1)

    if s == 1:
        return _dsd_sampling_01_B_C(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C(v, B, rng) for _ in range(s)]
        # stack as columns like R's replicate (N x s)
        return np.column_stack(samples)


def _dsd_sampling_01_B_C(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Real-valued version of .dsd_sampling_01_B_C in R.
    """
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    # Step 1: first element
    w = v.copy()

    pi1 = np.einsum("ij,ij->i", v, v)  # diag(v %*% t(v))
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (real case).")
        total += pi1[i] / n
    echant[i] = 1

    l = v[i, :]
    norm_l = np.sqrt(np.dot(l, l))
    if norm_l == 0:
        raise ValueError("Encountered zero-norm vector in real DSD.")
    e1 = l / norm_l

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)
        inter = v @ e1  # (N,)
        pi1 = pi1 - inter * inter
        pi2 = pi1 / r

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (real case).")
            total += pi2[i]
        echant[i] = 1

        # Update w and e1 (Gram-Schmidt-like update)
        proj = w @ e1                      # shape (N,)
        w = w - np.outer(proj, e1)         # (N,n)
        L = w[i, :]
        norm_L = np.sqrt(np.dot(L, L))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in real DSD.")
        e1 = L / norm_L

    if B:
        # 0/1 vector -> same as R "echant"
        return echant
    else:
        # indices (1-based, like in R: (1:N)[echant==1])
        return np.nonzero(echant == 1)[0] + 1


# -------------------- helpers: complex case --------------------- #

def _dsd_sampling_mult_complex(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    if s == 1:
        return _dsd_sampling_01_B_C_complex(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C_complex(v, B, rng) for _ in range(s)]
        return np.column_stack(samples)


def _dsd_sampling_01_B_C_complex(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Complex-valued version of .dsd_sampling_01_B_C_complex in R.
    """
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    w = v.copy()

    # pi1 = Re( diag( v %*% t(Conj(v)) ) )
    pi1 = np.real(np.einsum("ij,ij->i", v, np.conjugate(v)))

    if np.any(pi1 < 0) or np.any(pi1 >= 1):
        raise ValueError(
            "The matrix v given as input doesn't suit the expected input "
            "(cf. pgd / periodic_dsd)."
        )

    # Step 1: first element
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (complex case).")
        total += pi1[i] / n
    echant[i] = 1

    M = v[i, :]
    norm_M = np.sqrt(np.real(np.vdot(M, M)))
    if norm_M == 0:
        raise ValueError("Encountered zero-norm vector in complex DSD.")
    e1 = M / norm_M

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)

        # inter <- v %*% Conj(e1)
        inter = v @ np.conjugate(e1)           # (N,)
        pi1 = pi1 - np.real(inter * np.conjugate(inter))
        pi2 = np.real(pi1 / r)

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (complex case).")
            total += pi2[i]
        echant[i] = 1

        # w <- w - (projection on e1)
        proj = w @ np.conjugate(e1)           # (N,)
        w = w - np.outer(proj, e1)            # (N, n)

        L = w[i, :]
        norm_L = np.sqrt(np.real(np.vdot(L, L)))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in complex DSD.")
        e1 = L / norm_L

    if B:
        return echant
    else:
        # Return 1-based indices, to match the R function
        return np.nonzero(echant == 1)[0] + 1

In [2]:
def check_Ppi(Pi, V, tol=1e-7):
    Pi = np.asarray(Pi, dtype=float).ravel()
    K = V @ V.T

    out = {
        "N": len(Pi),
        "n": V.shape[1],
        "orthogonality_error": np.max(np.abs(V.T @ V - np.eye(V.shape[1]))),
        "diag_error": np.max(np.abs(np.diag(K) - Pi)),
        "projection_error": np.max(np.abs(K @ K - K)),
        "trace_K": np.trace(K),
        "sum_Pi": Pi.sum(),
    }

    out["ok"] = (
        out["orthogonality_error"] < tol
        and out["diag_error"] < tol
        and out["projection_error"] < tol
    )

    return out

In [3]:
# ============================================================
# CELL 2: Fast CaDsd kernel construction
# ============================================================
# This cell defines the fast Python implementation of the CaDsd
# construction used to generate candidate determinantal kernels.
#
# Main functions:
#   1. spec(omega, M, pi, spectre)
#      Builds the spectrum matrix used inside the CaDsd construction.
#
#   2. CaDsd(pi, M, omega, rho, spectre, U, option)
#      Builds a Hermitian projection-like DSD kernel K from:
#          - pi    : first-order inclusion probabilities,
#          - M     : sample size,
#          - omega : parameters controlling the spectrum path,
#          - rho   : phase/rotation parameters.
#
# Important note:
#   CaDsd internally sorts pi in decreasing order. Therefore, the
#   returned kernel K corresponds to the descending-pi order, not
#   necessarily to the original population order.
#
# After constructing K, we should check:
#   - diag(K) ≈ sorted(pi, decreasing=True)
#   - K is approximately Hermitian
#   - K is approximately a projection matrix
#   - trace(K) ≈ M
# ============================================================
import numpy as np
from typing import Optional, Union


def spec(omega: np.ndarray, M: int, pi: Union[np.ndarray, list], spectre=100) -> np.ndarray:
    """
    Faster Python/Numpy port of the R function spec().

    The output is the same object as before: a matrix of shape (M, N).
    Speed-up comes from cumulative sums instead of thousands of repeated
    small numpy sum calls inside Python loops.
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size
    omega = np.asarray(omega, dtype=float)
    if omega.shape != (M, N):
        raise ValueError("omega must have shape (M, N)")

    mat_spectre = np.zeros((M, N), dtype=float)
    pi_down = np.sort(pi)[::-1]
    pi_prefix = np.concatenate(([0.0], np.cumsum(pi_down)))
    mu = float(pi.sum())

    if (np.isscalar(spectre) and spectre == 100) or (
        not np.isscalar(spectre)
        and len(np.atleast_1d(spectre)) == 1
        and np.atleast_1d(spectre)[0] == 100
    ):
        spectre_vec = np.zeros(M, dtype=float)
        cumsum_1 = 0.0
        lam = 0.0

        for j in range(M - 1):
            jR = j + 1
            A = max(lam, mu - cumsum_1 - (M - jR))
            B = 1.0

            for iR in range(1, M - jR + 1):
                t_idx = M - jR - iR + 1
                s_down = pi_prefix[t_idx]
                val = (mu - cumsum_1 - s_down) / iR
                if val < B:
                    B = val

            val = (mu - cumsum_1) / (M - jR + 1)
            if val < B:
                B = val

            spectre_vec[j] = A + omega[j, N - 1] * (B - A)
            lam = spectre_vec[j]
            cumsum_1 += spectre_vec[j]

        spectre_vec[M - 1] = mu - spectre_vec[: M - 1].sum()
    else:
        spectre_arr = np.asarray(spectre, dtype=float).ravel()
        if spectre_arr.size != M:
            raise ValueError("spectre must have length M")
        spectre_vec = spectre_arr

    mat_spectre[:, N - 1] = spectre_vec

    for kR in range(N - 1, 0, -1):
        startR = max(1, M - kR + 1)
        lambda1 = mat_spectre[:, kR].copy()
        lambda2 = mat_spectre[:, kR - 1].copy()
        prefix_l1 = np.concatenate(([0.0], np.cumsum(lambda1)))
        cum_l2_before_j = 0.0

        # rows before startR are zero by construction, so cum_l2_before_j starts at 0.
        for jR in range(startR, M + 1):
            lam_prev = lambda1[jR - 2] if jR >= 2 else 0.0
            sum_l1 = prefix_l1[jR]
            A = max(0.0, lam_prev, sum_l1 - cum_l2_before_j - pi_down[kR])

            B_inner = float("inf")
            for iR in range(jR, M + 1):
                start_idx0 = M - iR
                if start_idx0 < kR:
                    prem = pi_prefix[kR] - pi_prefix[start_idx0]
                else:
                    prem = 0.0

                if jR <= (iR - 1):
                    deux = prefix_l1[iR - 1] - prefix_l1[jR - 1]
                else:
                    deux = 0.0

                B_val = prem - deux - cum_l2_before_j
                if B_val < B_inner:
                    B_inner = B_val

            B = min(lambda1[jR - 1], B_inner)
            new_val = A + omega[jR - 1, kR - 1] * (B - A)
            mat_spectre[jR - 1, kR - 1] = new_val
            lambda2[jR - 1] = new_val
            cum_l2_before_j += new_val

    return mat_spectre


def CaDsd(
    pi: Union[np.ndarray, list],
    M: Optional[int] = None,
    omega: Optional[np.ndarray] = None,
    rho: Optional[np.ndarray] = None,
    spectre=100,
    U: Optional[np.ndarray] = None,
    option: bool = True,
):
    """
    Faster CaDsd implementation with the same interface and output keys.

    Main speed-ups:
    - uses the faster spec() above;
    - preallocates phi instead of repeated hstack;
    - replaces permutation matrices by direct column indexing;
    - replaces multiplication by a diagonal phase matrix with column scaling.
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size

    if M is None:
        M = int(round(pi.sum(), 7))
    if int(M) != M:
        raise ValueError("M should be an integer")
    M = int(M)

    if omega is None:
        omega = 0.5 * np.ones((M, N), dtype=float)
    else:
        omega = np.asarray(omega, dtype=float)
        if omega.shape != (M, N):
            raise ValueError("omega must have shape (M, N)")

    if rho is None:
        rho = 0.5 * np.ones((M, N - 1), dtype=float)
    else:
        rho = np.asarray(rho, dtype=float)
        if rho.shape != (M, N - 1):
            raise ValueError("rho must have shape (M, N-1)")

    rho_angles = np.round(rho * 2 * np.pi, 7)
    mat_spectre = np.round(spec(omega, M, pi, spectre), 7)
    pi_down = np.sort(pi)[::-1]

    if U is None:
        U_work = np.eye(M, dtype=complex)
    else:
        U_work = np.asarray(U, dtype=complex).copy()
        if U_work.shape != (M, M):
            raise ValueError("U must have shape (M, M)")

    phi = np.zeros((M, N), dtype=complex)
    phi[:, 0] = np.round(np.sqrt(pi_down[0]) * U_work[:, 0], 7)
    ens = np.arange(1, M + 1, dtype=int)

    eye_index = np.arange(M)

    for kR in range(2, N + 1):
        phases = np.exp(1j * rho_angles[:, kR - 2])
        lambda1 = mat_spectre[:, kR - 1].copy()
        lambda2 = mat_spectre[:, kR - 2].copy()

        E1 = ens.tolist()
        E2 = ens.tolist()

        # Keep exact equality because mat_spectre is rounded to 7 digits as in the earlier code.
        for jR in ens:
            if not E1:
                break
            val = lambda2[jR - 1]
            lam1_E1 = lambda1[np.array(E1) - 1]
            matches = np.where(lam1_E1 == val)[0]
            if matches.size > 0:
                E2 = [x for x in E2 if x != jR]
                del E1[matches[0]]

        E1_arr = np.array(E1, dtype=int)
        E2_arr = np.array(E2, dtype=int)
        E1_rev = M + 1 - E1_arr
        E2_rev = M + 1 - E2_arr
        r = len(E1_rev)

        if r == 0:
            continue

        if r != M:
            mask1 = np.ones(M, dtype=bool)
            mask2 = np.ones(M, dtype=bool)
            mask1[E1_rev - 1] = False
            mask2[E2_rev - 1] = False
            E1_perm = np.concatenate([np.sort(E1_rev), ens[mask1]])
            E2_perm = np.concatenate([np.sort(E2_rev), ens[mask2]])
            perm1 = E1_perm - 1
            perm2 = E2_perm - 1
        else:
            perm1 = eye_index
            perm2 = eye_index

        lambda2_E2 = lambda2[E2_arr - 1]
        lambda1_E1 = lambda1[E1_arr - 1]
        R = np.column_stack([lambda2_E2, lambda1_E1]).astype(complex)[::-1, :]

        v = np.zeros(r, dtype=complex)
        w = np.zeros(r, dtype=complex)

        for i in range(r):
            v1 = R[i, 0] - R[:, 1]
            v2 = R[i, 0] - R[:, 0]
            v2[i] = 1.0 + 0j
            order = np.argsort(np.abs(v1))
            v[i] = np.round(np.sqrt(-np.prod(v1[order] / v2[order])), 7)

            w1 = R[i, 1] - R[:, 0]
            w2 = R[i, 1] - R[:, 1]
            w2[i] = 1.0 + 0j
            order_w = np.argsort(np.abs(w1))
            w[i] = np.round(np.sqrt(np.prod(w1[order_w] / w2[order_w])), 7)

        col_vals = R[:, 1]
        row_vals = R[:, 0]
        denom = col_vals[np.newaxis, :] - row_vals[:, np.newaxis]
        W = (v[:, None] * w[None, :]) / denom

        # U @ diag(phases) is just column scaling.
        UV = U_work * phases[np.newaxis, :]

        # Equivalent to U %*% V %*% t(sigma2) %*% vect.
        UV_sigma2 = UV[:, perm2]
        phi[:, kR - 1] = UV_sigma2[:, :r] @ v

        # Equivalent to U %*% V %*% t(sigma2) %*% block %*% sigma1.
        U_block = UV_sigma2.copy()
        U_block[:, :r] = UV_sigma2[:, :r] @ W
        U_new = np.empty_like(U_block)
        U_new[:, perm1] = U_block
        U_work = U_new

    d = np.sqrt(mat_spectre[:, N - 1])
    if np.any(d == 0):
        raise ValueError("Zero on diagonal of spectrum, cannot invert sqrt.")

    # Avoid forming an explicit inverse diagonal matrix.
    EigenBasis = (phi.conj().T @ U_work) / d[np.newaxis, :]
    K = np.round(phi.conj().T @ phi, 7)

    return {
        "K": K,
        "spectrum": mat_spectre,
        "EigenBasis": EigenBasis,
    }


print("Fast spec() and CaDsd() loaded.")


Fast spec() and CaDsd() loaded.


In [4]:
def check_CaDsd_output(pi, M, omega=None, rho=None, tol=1e-5):
    out = CaDsd(pi=pi, M=M, omega=omega, rho=rho)
    K = out["K"]

    pi_down = np.sort(np.asarray(pi, dtype=float))[::-1]

    checks = {
        "diag_error_desc_pi": np.max(np.abs(np.real(np.diag(K)) - pi_down)),
        "hermitian_error": np.max(np.abs(K - K.conj().T)),
        "projection_error": np.max(np.abs(K @ K - K)),
        "trace_K": np.real(np.trace(K)),
        "target_M": M,
    }

    checks["ok"] = (
        checks["diag_error_desc_pi"] < tol
        and checks["hermitian_error"] < tol
        and checks["projection_error"] < 1e-4
    )

    return checks

In [5]:
# ============================================================
# CELL 3: Inclusion probabilities
# ============================================================
# This cell defines inclusionprobabilities(p, n), which converts
# a positive size measure p into first-order inclusion probabilities
# pi with:
#
#     0 < pi_i <= 1
#     sum(pi) = n
#
# If a package version is available, we use it. Otherwise, we use
# the local fallback implementation below.
# ============================================================

import numpy as np

# --------------------------------------------------------
# 1) inclusionprobabilities() – Python version
#    (size measure p -> inclusion probs π with sum π = n)
# --------------------------------------------------------

def inclusionprobabilities(p, n):
    """
    Rough equivalent of sampling::inclusionprobabilities(p, n) in R.

    p : array-like, strictly positive size measures
    n : desired fixed sample size (integer)

    Returns
    -------
    pik : ndarray of length N
        First-order inclusion probabilities, 0 < pik_i <= 1, sum pik_i = n.
    """
    p = np.asarray(p, dtype=float)
    N = p.size
    if n <= 0 or n > N:
        raise ValueError("n must be in 1..N")

    pik = np.zeros(N, dtype=float)
    mask_fixed = np.zeros(N, dtype=bool)  # True where pik is already fixed to 1
    n_rem = float(n)
    p_work = p.copy()

    # Iterative rescaling: set any prob >= 1 to 1, rescale remaining
    while True:
        idx = ~mask_fixed
        if not np.any(idx):
            break

        p_sub = p_work[idx]
        # if all remaining p_sub are zero but n_rem > 0, it's impossible
        if p_sub.sum() <= 0 and n_rem > 1e-12:
            raise RuntimeError("Cannot construct inclusion probabilities with given p and n.")

        temp = n_rem * p_sub / p_sub.sum()
        over = temp >= (1.0 - 1e-12)  # tolerance

        # If no temp >= 1, we're done
        if not np.any(over):
            pik[idx] = temp
            break

        # Fix those >=1 to exactly 1
        idx_global = np.where(idx)[0]
        over_global = idx_global[over]

        pik[over_global] = 1.0
        mask_fixed[over_global] = True
        n_rem -= over.sum()
        p_work[over_global] = 0.0

        if n_rem <= 1e-12:
            # All remaining inclusion probabilities must be zero
            break

    return pik




In [6]:
# # ============================================================
# # OPTIONAL CELL: Python-to-R conversion helpers
# # ============================================================
# # These helper functions convert Python/Numpy vectors and matrices
# # into R code strings.
# #
# # They are only needed if we want to pass objects such as pi, K, V,
# # omega, or rho to R for comparison with the original R functions.
# #
# # In the main Python-only ABC workflow, these functions are not used.
# # If no later cell calls to_r_vector(), to_r_matrix_real(), or
# # to_r_matrix_complex(), this cell can be safely ignored.
# # ============================================================

# def to_r_vector(arr, name="vec"):
#     arr = np.asarray(arr).ravel()
#     values = ", ".join([f"{x:.10f}" for x in arr])
#     return f"{name} <- c({values})"

# def to_r_matrix_complex(mat, name="mat"):
#     mat = np.asarray(mat)
#     N, M = mat.shape
    
#     r_code = f"{name} <- matrix(c(\n"
#     elements = []
    
#     for j in range(M):  # R fills by column
#         for i in range(N):
#             val = mat[i, j]
#             real_part = np.real(val)
#             imag_part = np.imag(val)
            
#             if abs(imag_part) < 1e-10:
#                 elements.append(f"  {real_part:.10f}")
#             else:
#                 sign = "+" if imag_part >= 0 else ""
#                 elements.append(f"  complex(real={real_part:.10f}, imaginary={imag_part:.10f})")
    
#     r_code += ",\n".join(elements)
#     r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
#     return r_code

# def to_r_matrix_real(mat, name="mat"):
#     mat = np.asarray(mat)
#     N, M = mat.shape
    
#     r_code = f"{name} <- matrix(c(\n"
#     elements = []
    
#     for j in range(M):  # R fills by column
#         for i in range(N):
#             elements.append(f"  {mat[i, j]:.10f}")
    
#     r_code += ",\n".join(elements)
#     r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
#     return r_code


In [7]:
# ============================================================
# PRECHECK HELPER FUNCTIONS ONLY
# ============================================================
# Put this cell near the beginning.
# It only defines functions. It does NOT run the precheck.
# ============================================================

RUN_PRECHECK = True
CADSD_PI_ATOL = 1e-3


def ppi_efficiency_for_order(y_ordered, z_ordered, pi_ordered, var_srs_y, var_srs_z):
    y_ordered = np.asarray(y_ordered, dtype=float).ravel()
    z_ordered = np.asarray(z_ordered, dtype=float).ravel()
    pi_ordered = np.asarray(pi_ordered, dtype=float).ravel()

    V = Ppi(pi_ordered)
    K = V @ V.T

    diag_error = float(np.max(np.abs(np.diag(K) - pi_ordered)))

    A = -np.abs(K) ** 2
    np.fill_diagonal(A, pi_ordered * (1.0 - pi_ordered))

    y_over_pi = y_ordered / pi_ordered
    z_over_pi = z_ordered / pi_ordered

    var_y = float(np.real(y_over_pi @ (A @ y_over_pi)))
    var_z = float(np.real(z_over_pi @ (A @ z_over_pi)))

    eff_y = var_srs_y / var_y if var_y > 0 else np.nan
    eff_z = var_srs_z / var_z if var_z > 0 else np.nan

    return {
        "eff_y": eff_y,
        "eff_z": eff_z,
        "diag_error": diag_error,
    }


def precheck_one_case(df, y_name, z_name, x_name, n):
    y_raw = df[y_name].to_numpy(dtype=float)
    z_raw = df[z_name].to_numpy(dtype=float)
    x_raw = df[x_name].to_numpy(dtype=float)

    N = len(df)
    pi_raw = inclusionprobabilities(x_raw, n)

    var_srs_y = N**2 * (1.0 - n / N) * np.var(y_raw, ddof=1) / n
    var_srs_z = N**2 * (1.0 - n / N) * np.var(z_raw, ddof=1) / n

    idx_vincent = np.argsort(z_raw / pi_raw)

    y_v = y_raw[idx_vincent]
    z_v = z_raw[idx_vincent]
    pi_v = pi_raw[idx_vincent]

    vincent_direct = ppi_efficiency_for_order(
        y_v,
        z_v,
        pi_v,
        var_srs_y,
        var_srs_z,
    )

    probe = ABCAlgorithm(
        y_sorted=y_v,
        z_sorted=z_v,
        pik_sorted=pi_v,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        M=n,
        n=n,
        case_name=f"PRECHECK_{z_name}_{x_name}_n{n}",
        objective=OBJECTIVE,
        enforce_cadsd_order=True,
        random_state=ABC_RANDOM_SEED,
        validation_mode="fast",
        eigen_check_interval=0,
        initial_strict_checks=2,
    )

    abc_matches_vincent_z = np.allclose(
        probe.eff_z_optimal,
        vincent_direct["eff_z"],
        atol=1e-8,
        rtol=1e-8,
    )

    same_pi_multiset = np.allclose(
        np.sort(probe.input_pi),
        np.sort(probe.pik_sorted),
        atol=1e-12,
        rtol=1e-12,
    )

    K_center = CaDsd(
        pi=pi_v,
        M=n,
        omega=0.5 * np.ones((n, N)),
        rho=0.5 * np.ones((n, N - 1)),
    )["K"]

    cadsd_diag_error = float(
        np.max(np.abs(np.sort(np.real(np.diag(K_center))) - np.sort(pi_raw)))
    )

    return {
        "z": z_name,
        "x_for_pi": x_name,
        "n": n,
        "sum_pi": pi_raw.sum(),
        "ABC_matches_Vincent_z": abc_matches_vincent_z,
        "same_pi_multiset": same_pi_multiset,
        "CaDsd_same_pi": cadsd_diag_error <= CADSD_PI_ATOL,
        "Ppi_eff_z_direct": vincent_direct["eff_z"],
        "Ppi_eff_z_probe": probe.eff_z_optimal,
        "Ppi_diag_error": vincent_direct["diag_error"],
        "CaDsd_diag_error": cadsd_diag_error,
    }


print("Precheck helper functions loaded.")

Precheck helper functions loaded.


## ABC Algorithm


In [8]:
# ============================================================
# CELL 4: ABC optimizer and random-search baseline
# ============================================================
# This cell defines the main optimization classes:
#
#   1. ABCAlgorithm
#      Artificial Bee Colony optimizer for the CaDsd omega/rho
#      parameterization. It searches for omega and rho values that
#      produce a DSD kernel K with high efficiency for z.
#
#   2. RandomSearchAlgorithm
#      A pure random-search baseline using the same CaDsd construction
#      and the same efficiency calculation.
#
# Important ordering note:
#   The input y, z, and pi supplied to ABCAlgorithm should already be
#   ordered by Vincent's empirical order, i.e. by z/pi.
#
#   Internally, ABCAlgorithm reorders y, z, and pi by decreasing pi
#   before evaluating CaDsd kernels, because CaDsd works in decreasing-pi
#   order.
#
# Important reference note:
#   _calculate_optimal() computes the true Vincent Pπ reference using the
#   original input order, which must be z/pi order.
#
#   The old internal decreasing-pi Pπ reference is removed, because it was
#   unnecessary and could fail for some pi orders.
#
# Recommended validation:
#   Use validation_mode="fast" for normal runs.
#   Use validation_mode="strict" only for debugging or checking final kernels.
# ============================================================

import numpy as np
import time
from typing import Tuple, List, Optional


class ABCAlgorithm:
    """
    Artificial Bee Colony optimizer for the CaDsd parameterization.

    The optimizer searches over omega and rho. For each candidate, CaDsd
    constructs a DSD kernel K, and the HT variance is computed for y and z.

    The reference design is Vincent's Pπ design in z/pi order.
    """

    def __init__(
        self,
        y_sorted,
        z_sorted,
        pik_sorted,
        var_srs_y,
        var_srs_z,
        M,
        n,
        case_name="",
        objective: str = "eff_z",
        enforce_cadsd_order: bool = True,
        random_state: Optional[int] = None,
        validation_mode: str = "fast",
        eigen_check_interval: int = 0,
        initial_strict_checks: int = 2,
    ):
        self.rng = np.random.default_rng(random_state)

        # ----------------------------------------------------
        # Input order: this should be Vincent z/pi order.
        # ----------------------------------------------------
        self.input_y = np.asarray(y_sorted, dtype=float).ravel()
        self.input_z = np.asarray(z_sorted, dtype=float).ravel()
        self.input_pi = np.asarray(pik_sorted, dtype=float).ravel()

        if not (len(self.input_y) == len(self.input_z) == len(self.input_pi)):
            raise ValueError("y_sorted, z_sorted, and pik_sorted must have the same length.")

        if np.any(self.input_pi <= 0):
            raise ValueError("All inclusion probabilities must be positive.")

        if np.any(self.input_pi >= 1):
            raise ValueError("This implementation requires all inclusion probabilities to be < 1.")

        self.enforce_cadsd_order = enforce_cadsd_order

        # ----------------------------------------------------
        # Internal CaDsd order: decreasing pi.
        # ----------------------------------------------------
        if enforce_cadsd_order:
            self.order = np.argsort(self.input_pi)[::-1]
        else:
            self.order = np.arange(len(self.input_pi))

        self.y_sorted = self.input_y[self.order]
        self.z_sorted = self.input_z[self.order]
        self.pik_sorted = self.input_pi[self.order]

        self.var_srs_y = float(var_srs_y)
        self.var_srs_z = float(var_srs_z)
        self.M = int(M)
        self.n = int(n)
        self.N = len(self.pik_sorted)
        self.case_name = case_name
        self.objective = objective

        if validation_mode not in {"fast", "strict"}:
            raise ValueError("validation_mode must be 'fast' or 'strict'.")

        self.validation_mode = validation_mode
        self.eigen_check_interval = int(eigen_check_interval) if eigen_check_interval else 0
        self.initial_strict_checks = int(initial_strict_checks)

        # Compatibility objects for later sensitivity functions.
        self.I_N = np.eye(self.N)
        self.Dpi_inv = np.diag(1.0 / self.pik_sorted)

        # Fast HT variance vectors in internal CaDsd order.
        self.y_over_pi = self.y_sorted / self.pik_sorted
        self.z_over_pi = self.z_sorted / self.pik_sorted

        # CaDsd should return a kernel whose diagonal is decreasing-pi.
        self.pik_sorted_desc = np.sort(self.input_pi)[::-1]

        # Check that internal ordering did not change the pi multiset.
        self.same_pi_multiset_check = np.allclose(
            np.sort(self.input_pi),
            np.sort(self.pik_sorted),
            atol=1e-12,
            rtol=1e-12,
        )

        # Reference Pπ design.
        self._calculate_optimal()

        # Best solution tracking.
        self.global_best_eff_z = 0.0
        self.global_best_eff_y = 0.0
        self.global_best_score = -np.inf
        self.global_best_omega = None
        self.global_best_rho = None

        # History and counters.
        self.history = []
        self.history_records = []
        self.scout_history = []
        self.eval_count = 0
        self.valid_count = 0
        self.best_update_count = 0
        self.strict_check_count = 0

    def _variance_pair_from_kernel(
        self,
        Kmat: np.ndarray,
        diag_K: Optional[np.ndarray] = None,
    ) -> Tuple[float, float]:
        """
        Fast HT variance calculation for a DSD kernel in internal CaDsd order.
        """
        if diag_K is None:
            diag_K = np.real(np.diag(Kmat))

        A = -np.abs(Kmat) ** 2
        np.fill_diagonal(A, diag_K * (1.0 - diag_K))

        var_y = float(np.real(self.y_over_pi @ (A @ self.y_over_pi)))
        var_z = float(np.real(self.z_over_pi @ (A @ self.z_over_pi)))

        return var_y, var_z

    def _variance_from_kernel_given_order(self, Kmat, y, z, pi):
        """
        Compute HT variances for a kernel K using the same order as y, z, and pi.
        """
        y = np.asarray(y, dtype=float).ravel()
        z = np.asarray(z, dtype=float).ravel()
        pi = np.asarray(pi, dtype=float).ravel()

        diag_K = np.real(np.diag(Kmat))

        A = -np.abs(Kmat) ** 2
        np.fill_diagonal(A, diag_K * (1.0 - diag_K))

        y_over_pi = y / pi
        z_over_pi = z / pi

        var_y = float(np.real(y_over_pi @ (A @ y_over_pi)))
        var_z = float(np.real(z_over_pi @ (A @ z_over_pi)))

        return var_y, var_z

    def _calculate_optimal(self):
        """
        Compute the true Vincent Pπ reference.

        Important:
        self.input_y, self.input_z, and self.input_pi must already be
        ordered by z/pi before entering this class.
        """
        Base_vincent = Ppi(self.input_pi)
        K_vincent = Base_vincent @ Base_vincent.T

        diag_vincent = np.real(np.diag(K_vincent))

        self.vincent_ppi_diag_error = float(
            np.max(np.abs(diag_vincent - self.input_pi))
        )

        self.vincent_ppi_projection_error = float(
            np.max(np.abs(K_vincent @ K_vincent - K_vincent))
        )

        var_vincent_y, var_vincent_z = self._variance_from_kernel_given_order(
            K_vincent,
            self.input_y,
            self.input_z,
            self.input_pi,
        )

        self.var_opt_y = var_vincent_y
        self.var_opt_z = var_vincent_z

        self.eff_y_optimal = (
            self.var_srs_y / var_vincent_y
            if var_vincent_y > 0
            else np.inf
        )

        self.eff_z_optimal = (
            self.var_srs_z / var_vincent_z
            if var_vincent_z > 0
            else np.inf
        )

        # Old internal Pπ diagnostic is removed.
        # These placeholders prevent downstream KeyError if older cells refer to them.
        self.var_ppi_internal_y = np.nan
        self.var_ppi_internal_z = np.nan
        self.eff_y_ppi_internal = np.nan
        self.eff_z_ppi_internal = np.nan
        self.internal_ppi_diag_error = np.nan

    def _score(self, eff_z: float, eff_y: float) -> float:
        """Objective used by ABC."""
        if self.objective == "eff_z":
            return eff_z

        if self.objective == "eff_y":
            return eff_y

        if self.objective == "harmonic":
            if eff_z <= 0 or eff_y <= 0:
                return 0.0
            return 2.0 * eff_z * eff_y / (eff_z + eff_y)

        if self.objective == "mean":
            return 0.5 * (eff_z + eff_y)

        raise ValueError("objective must be one of: 'eff_z', 'eff_y', 'harmonic', 'mean'.")

    def _needs_strict_kernel_check(self) -> bool:
        if self.validation_mode == "strict":
            return True

        if self.eval_count <= self.initial_strict_checks:
            return True

        if self.eigen_check_interval and self.eval_count % self.eigen_check_interval == 0:
            return True

        return False

    def _kernel_passes_validation(self, Kmat: np.ndarray, diag_K: np.ndarray) -> bool:
        if not np.all(np.isfinite(diag_K)):
            return False

        if not np.allclose(diag_K, self.pik_sorted_desc, atol=1e-3):
            return False

        # In fast mode, CaDsd is trusted after the diagonal check.
        if not self._needs_strict_kernel_check():
            return True

        self.strict_check_count += 1

        try:
            evals = np.linalg.eigvalsh(Kmat)
        except Exception:
            return False

        if not (np.all(evals >= -1e-3) and np.all(evals <= 1.0 + 1e-3)):
            return False

        if not np.isclose(evals.sum(), self.n, atol=1e-3):
            return False

        return True

    def evaluate(self, omega: np.ndarray, rho: np.ndarray) -> Tuple[float, float, bool]:
        """
        Evaluate one candidate solution and return eff_z, eff_y, valid.
        """
        self.eval_count += 1

        try:
            K_dict = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega, rho=rho)
            Kmat = K_dict["K"].astype(np.complex128, copy=False)
        except Exception:
            return (0.0, 0.0, False)

        diag_K = np.real(np.diag(Kmat))

        if not self._kernel_passes_validation(Kmat, diag_K):
            return (0.0, 0.0, False)

        var_y, var_z = self._variance_pair_from_kernel(Kmat, diag_K=diag_K)

        if (
            not np.isfinite(var_y)
            or not np.isfinite(var_z)
            or var_y <= 0
            or var_z <= 0
        ):
            return (0.0, 0.0, False)

        eff_y = self.var_srs_y / var_y
        eff_z = self.var_srs_z / var_z

        if not (np.isfinite(eff_y) and np.isfinite(eff_z)):
            return (0.0, 0.0, False)

        self.valid_count += 1

        return (eff_z, eff_y, True)

    def _food_from_arrays(
        self,
        omega: np.ndarray,
        rho: np.ndarray,
        trial: int = 0,
    ) -> Optional[dict]:
        eff_z, eff_y, valid = self.evaluate(omega, rho)

        if not valid:
            return None

        score = self._score(eff_z, eff_y)

        food = {
            "omega": omega,
            "rho": rho,
            "eff_z": eff_z,
            "eff_y": eff_y,
            "score": score,
            "trial": trial,
        }

        self._update_global_best(food)

        return food

    def _update_global_best(self, food: dict) -> bool:
        if food["score"] > self.global_best_score:
            self.global_best_score = food["score"]
            self.global_best_eff_z = food["eff_z"]
            self.global_best_eff_y = food["eff_y"]
            self.global_best_omega = food["omega"].copy()
            self.global_best_rho = food["rho"].copy()
            self.best_update_count += 1
            return True

        return False

    def _random_candidate(self) -> Tuple[np.ndarray, np.ndarray]:
        omega = self.rng.random((self.M, self.N))
        rho = self.rng.random((self.M, self.N - 1))

        return omega, rho

    def _candidate_near_best(self, scale: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
        if self.global_best_omega is None or self.global_best_rho is None:
            return self._random_candidate()

        omega = self.global_best_omega + self.rng.normal(
            0.0,
            scale,
            self.global_best_omega.shape,
        )

        rho = self.global_best_rho + self.rng.normal(
            0.0,
            scale,
            self.global_best_rho.shape,
        )

        return np.clip(omega, 0.0, 1.0), np.clip(rho, 0.0, 1.0)

    def initialize_population(self, colony_size: int, verbose: bool = True) -> List[dict]:
        """Initialize food sources."""
        if verbose:
            print(f"Initializing {colony_size} food sources...")

        population = []

        center_omega = 0.5 * np.ones((self.M, self.N))
        center_rho = 0.5 * np.ones((self.M, self.N - 1))

        food = self._food_from_arrays(center_omega, center_rho)

        if food is not None:
            population.append(food)

        max_attempts = max(colony_size * 20, 120)
        count = 0

        while len(population) < colony_size and count < max_attempts:
            omega, rho = self._random_candidate()
            food = self._food_from_arrays(omega, rho)

            if food is not None:
                population.append(food)

            count += 1

            if verbose and count % 100 == 0:
                print(
                    f"  evaluated {count}, valid sources {len(population)}/{colony_size}",
                    end="\r",
                )

        if verbose:
            print(f"\nInitialized {len(population)} food sources.")
            if len(population) > 0:
                print(
                    f"Best initial: eff_z={self.global_best_eff_z:.4f}, "
                    f"eff_y={self.global_best_eff_y:.4f}"
                )
            else:
                print("WARNING: no valid solution found.")

        return population

    def _mutate_food(
        self,
        food: dict,
        partner: dict,
        progress: float,
        mode: str = "abc",
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Create a new candidate with adaptive exploration."""
        adaptive_scale = max(0.12, 1.0 - 0.88 * progress)

        if mode == "local":
            return self._candidate_near_best(scale=0.015 + 0.07 * (1.0 - progress))

        phi_omega = self.rng.uniform(
            -adaptive_scale,
            adaptive_scale,
            food["omega"].shape,
        )

        phi_rho = self.rng.uniform(
            -adaptive_scale,
            adaptive_scale,
            food["rho"].shape,
        )

        new_omega = food["omega"] + phi_omega * (food["omega"] - partner["omega"])
        new_rho = food["rho"] + phi_rho * (food["rho"] - partner["rho"])

        if self.global_best_omega is not None and progress > 0.10:
            pull = self.rng.uniform(0.0, 0.30 * progress)
            new_omega = new_omega + pull * (self.global_best_omega - new_omega)
            new_rho = new_rho + pull * (self.global_best_rho - new_rho)

        jitter = 0.006 * adaptive_scale

        new_omega = new_omega + self.rng.normal(0.0, jitter, new_omega.shape)
        new_rho = new_rho + self.rng.normal(0.0, jitter, new_rho.shape)

        return np.clip(new_omega, 0.0, 1.0), np.clip(new_rho, 0.0, 1.0)

    def _greedy_replace(self, old_food: dict, omega: np.ndarray, rho: np.ndarray) -> dict:
        new_food = self._food_from_arrays(omega, rho)

        if new_food is not None and new_food["score"] > old_food["score"]:
            return new_food

        old_copy = old_food.copy()
        old_copy["trial"] = old_food["trial"] + 1

        return old_copy

    def employed_bee_phase(self, population: List[dict], progress: float) -> List[dict]:
        if len(population) == 0:
            return population

        new_population = []

        for i, food in enumerate(population):
            if len(population) > 1:
                k = int(self.rng.integers(0, len(population) - 1))
                if k >= i:
                    k += 1
            else:
                k = i

            new_omega, new_rho = self._mutate_food(
                food,
                population[k],
                progress,
                mode="abc",
            )

            new_population.append(
                self._greedy_replace(food, new_omega, new_rho)
            )

        return new_population

    def onlooker_bee_phase(
        self,
        population: List[dict],
        progress: float,
        onlooker_factor: float = 0.60,
    ) -> List[dict]:
        if len(population) == 0 or onlooker_factor <= 0:
            return population

        scores = np.array([food["score"] for food in population], dtype=float)
        scores = scores - np.min(scores) + 1e-12

        if not np.isfinite(scores).all() or scores.sum() <= 0:
            probs = np.ones(len(population)) / len(population)
        else:
            probs = scores / scores.sum()

        new_population = [food.copy() for food in population]
        n_onlookers = max(1, int(round(onlooker_factor * len(population))))

        for _ in range(n_onlookers):
            i = int(self.rng.choice(len(population), p=probs))

            if len(population) > 1:
                k = int(self.rng.integers(0, len(population) - 1))
                if k >= i:
                    k += 1
            else:
                k = i

            mode = "local" if self.rng.random() < 0.10 else "abc"

            new_omega, new_rho = self._mutate_food(
                new_population[i],
                population[k],
                progress,
                mode=mode,
            )

            new_population[i] = self._greedy_replace(
                new_population[i],
                new_omega,
                new_rho,
            )

        return new_population

    def scout_bee_phase(
        self,
        population: List[dict],
        limit: int,
        progress: float,
    ) -> Tuple[List[dict], int]:
        if len(population) == 0:
            return population, 0

        new_population = []
        n_abandoned = 0

        for food in population:
            if food["trial"] >= limit:
                n_abandoned += 1
                best_new = None
                attempts = 0
                max_attempts = 25

                while attempts < max_attempts:
                    if self.rng.random() < 0.60:
                        omega, rho = self._candidate_near_best(
                            scale=0.03 + 0.08 * (1.0 - progress)
                        )
                    else:
                        omega, rho = self._random_candidate()

                    candidate = self._food_from_arrays(omega, rho)

                    if candidate is not None and (
                        best_new is None or candidate["score"] > best_new["score"]
                    ):
                        best_new = candidate

                    attempts += 1

                if best_new is not None:
                    best_new["trial"] = 0
                    new_population.append(best_new)
                else:
                    food_copy = food.copy()
                    food_copy["trial"] = 0
                    new_population.append(food_copy)

            else:
                new_population.append(food)

        return new_population, n_abandoned

    def local_search_phase(
        self,
        population: List[dict],
        progress: float,
        attempts: int = 2,
    ) -> List[dict]:
        if len(population) == 0 or self.global_best_omega is None or attempts <= 0:
            return population

        best_candidate = None
        scale = max(0.004, 0.03 * (1.0 - progress))

        for _ in range(attempts):
            omega, rho = self._candidate_near_best(scale=scale)
            candidate = self._food_from_arrays(omega, rho)

            if candidate is not None and (
                best_candidate is None or candidate["score"] > best_candidate["score"]
            ):
                best_candidate = candidate

        if best_candidate is not None:
            worst_idx = int(np.argmin([food["score"] for food in population]))

            if best_candidate["score"] > population[worst_idx]["score"]:
                population[worst_idx] = best_candidate

        return population

    def _inject_elite(self, population: List[dict]) -> List[dict]:
        """Ensure the current global best is not lost."""
        if len(population) == 0 or self.global_best_omega is None:
            return population

        scores = np.array([food["score"] for food in population])

        if np.max(scores) + 1e-15 < self.global_best_score:
            worst_idx = int(np.argmin(scores))

            population[worst_idx] = {
                "omega": self.global_best_omega.copy(),
                "rho": self.global_best_rho.copy(),
                "eff_z": self.global_best_eff_z,
                "eff_y": self.global_best_eff_y,
                "score": self.global_best_score,
                "trial": 0,
            }

        return population

    def optimize(
        self,
        colony_size: int = 40,
        max_iterations: int = 100,
        limit: int = 25,
        verbose: bool = True,
        progress_interval: Optional[int] = None,
        local_search_interval: int = 25,
        local_search_attempts: int = 2,
        onlooker_factor: float = 0.60,
        early_stopping: bool = True,
        patience: Optional[int] = None,
        min_iterations: Optional[int] = None,
        rel_tol: float = 1e-5,
        time_limit_seconds: Optional[float] = None,
        random_searcher: Optional["RandomSearchAlgorithm"] = None,
        random_evals_per_iteration: Optional[int] = None,
        random_start_evals: Optional[int] = None,
    ) -> dict:
        """Run ABC, optionally with a side-by-side random-search baseline."""
        start_time = time.time()

        if progress_interval is None:
            progress_interval = max(1, max_iterations // 20)

        if patience is None:
            patience = max(25, max_iterations // 4)

        if min_iterations is None:
            min_iterations = max(20, max_iterations // 3)

        use_random = random_searcher is not None

        if use_random:
            if random_evals_per_iteration is None:
                random_evals_per_iteration = max(
                    1,
                    int(round(colony_size * (1.0 + onlooker_factor))),
                )

            if random_start_evals is None:
                random_start_evals = colony_size

        if verbose:
            print("=" * 80)
            print(f"ABC: {self.case_name}")
            print(
                f"Reference Pπ: eff_z={self.eff_z_optimal:.4f}, "
                f"eff_y={self.eff_y_optimal:.4f}"
            )
            print(
                f"ABC settings: colony={colony_size}, iter={max_iterations}, "
                f"limit={limit}, progress_every={progress_interval}"
            )
            if use_random:
                print(
                    f"Random search: start={random_start_evals}, "
                    f"per_iter={random_evals_per_iteration}"
                )
            print("-" * 80)

        population = self.initialize_population(colony_size, verbose)

        if len(population) == 0:
            if verbose:
                print("FAILED: could not initialize any valid solution.")
            return self._prepare_results(time.time() - start_time, colony_size, 0)

        if use_random and random_start_evals and random_start_evals > 0:
            random_searcher.step(random_start_evals, include_center_once=True)

        total_abandoned = 0
        start_best = self.global_best_eff_z
        last_improvement_iter = 0
        stopped_early = False
        stopped_by_time = False
        completed_iterations = 0

        if verbose:
            if use_random:
                header = (
                    f"{'iter':>6} | {'ABC_z':>10} | {'Rand_z':>10} | "
                    f"{'ABC/Pπ':>8} | {'Rand/Pπ':>8} | "
                    f"{'scout':>5} | {'valid/eval':>12} | {'elapsed':>8}"
                )
            else:
                header = (
                    f"{'iter':>6} | {'ABC_z':>10} | {'ABC_y':>10} | "
                    f"{'ABC/Pπ':>8} | {'scout':>5} | "
                    f"{'valid/eval':>12} | {'elapsed':>8}"
                )

            print(header)
            print("-" * len(header))

        for iteration in range(max_iterations):
            progress = (iteration + 1) / max_iterations
            old_best = self.global_best_score

            population = self.employed_bee_phase(population, progress)
            population = self.onlooker_bee_phase(
                population,
                progress,
                onlooker_factor=onlooker_factor,
            )

            population, n_abandoned = self.scout_bee_phase(
                population,
                limit,
                progress,
            )

            total_abandoned += n_abandoned

            if local_search_interval and (iteration + 1) % local_search_interval == 0:
                population = self.local_search_phase(
                    population,
                    progress,
                    attempts=local_search_attempts,
                )

            population = self._inject_elite(population)

            if use_random and random_evals_per_iteration and random_evals_per_iteration > 0:
                random_searcher.step(
                    random_evals_per_iteration,
                    include_center_once=False,
                )

            completed_iterations = iteration + 1

            if self.global_best_score > old_best * (1.0 + rel_tol):
                last_improvement_iter = iteration + 1

            self.history.append(self.global_best_eff_z)
            self.scout_history.append(n_abandoned)

            record = {
                "iteration": iteration + 1,
                "best_eff_z": self.global_best_eff_z,
                "best_eff_y": self.global_best_eff_y,
                "best_score": self.global_best_score,
                "n_abandoned": n_abandoned,
                "eval_count": self.eval_count,
                "valid_count": self.valid_count,
                "strict_check_count": self.strict_check_count,
                "elapsed": time.time() - start_time,
            }

            if use_random:
                record.update({
                    "random_best_eff_z": random_searcher.global_best_eff_z,
                    "random_best_eff_y": random_searcher.global_best_eff_y,
                    "random_best_score": random_searcher.global_best_score,
                    "random_eval_count": random_searcher.eval_count,
                    "random_valid_count": random_searcher.valid_count,
                })

            self.history_records.append(record)

            should_print = (
                verbose
                and (
                    iteration == 0
                    or (iteration + 1) % progress_interval == 0
                    or (iteration + 1) == max_iterations
                )
            )

            if should_print:
                abc_ratio = (
                    self.global_best_eff_z / self.eff_z_optimal
                    if self.eff_z_optimal > 0 and self.global_best_eff_z > 0
                    else np.nan
                )

                valid_ratio = f"{self.valid_count}/{self.eval_count}"
                elapsed = time.time() - start_time

                if use_random:
                    rand_ratio = (
                        random_searcher.global_best_eff_z / self.eff_z_optimal
                        if self.eff_z_optimal > 0 and random_searcher.global_best_eff_z > 0
                        else np.nan
                    )

                    print(
                        f"{iteration + 1:6d} | "
                        f"{self.global_best_eff_z:10.4f} | "
                        f"{random_searcher.global_best_eff_z:10.4f} | "
                        f"{abc_ratio:8.3f} | "
                        f"{rand_ratio:8.3f} | "
                        f"{n_abandoned:5d} | "
                        f"{valid_ratio:>12} | "
                        f"{elapsed:7.1f}s"
                    )

                else:
                    print(
                        f"{iteration + 1:6d} | "
                        f"{self.global_best_eff_z:10.4f} | "
                        f"{self.global_best_eff_y:10.4f} | "
                        f"{abc_ratio:8.3f} | "
                        f"{n_abandoned:5d} | "
                        f"{valid_ratio:>12} | "
                        f"{elapsed:7.1f}s"
                    )

            if (
                early_stopping
                and (iteration + 1) >= min_iterations
                and (iteration + 1 - last_improvement_iter) >= patience
            ):
                stopped_early = True
                if verbose:
                    print(
                        f"Early stop at iter {iteration + 1}: "
                        f"no relative improvement > {rel_tol:g} for {patience} iterations."
                    )
                break

            if (
                time_limit_seconds is not None
                and (iteration + 1) >= min_iterations
                and (time.time() - start_time) >= float(time_limit_seconds)
            ):
                stopped_by_time = True
                if verbose:
                    print(
                        f"Time stop at iter {iteration + 1}: "
                        f"reached {time_limit_seconds}s budget."
                    )
                break

        total_time = time.time() - start_time
        results = self._prepare_results(total_time, colony_size, total_abandoned)

        if use_random:
            random_relative_to_ppi = (
                100.0 * (random_searcher.global_best_eff_z / self.eff_z_optimal - 1.0)
                if self.eff_z_optimal > 0 and random_searcher.global_best_eff_z > 0
                else -np.inf
            )

            results.update({
                "random_best_eff_z": random_searcher.global_best_eff_z,
                "random_best_eff_y": random_searcher.global_best_eff_y,
                "random_best_score": random_searcher.global_best_score,
                "random_relative_to_ppi_percent": random_relative_to_ppi,
                "random_eval_count": random_searcher.eval_count,
                "random_valid_count": random_searcher.valid_count,
                "random_history": random_searcher.history.copy(),
                "random_history_records": random_searcher.random_history_records.copy(),
                "random_search_enabled": True,
                "random_evals_per_iteration": random_evals_per_iteration,
                "random_start_evals": random_start_evals,
            })
        else:
            results["random_search_enabled"] = False

        results["stopped_early"] = stopped_early
        results["stopped_by_time"] = stopped_by_time
        results["completed_iterations"] = completed_iterations
        results["strict_check_count"] = self.strict_check_count

        return results

    def _prepare_results(
        self,
        total_time: float,
        colony_size: int,
        total_abandoned: int,
    ) -> dict:
        relative_to_ppi = (
            100.0 * (self.global_best_eff_z / self.eff_z_optimal - 1.0)
            if self.eff_z_optimal > 0 and self.global_best_eff_z > 0
            else -np.inf
        )

        gap_percent = (
            100.0 * (self.eff_z_optimal / self.global_best_eff_z - 1.0)
            if self.eff_z_optimal > 0 and self.global_best_eff_z > 0
            else np.inf
        )

        success = self.global_best_eff_z > self.eff_z_optimal

        return {
            "case": self.case_name,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "best_omega": self.global_best_omega,
            "best_rho": self.global_best_rho,

            # Correct Vincent Pπ reference.
            "optimal_eff_z": self.eff_z_optimal,
            "optimal_eff_y": self.eff_y_optimal,

            # Backward-compatible aliases.
            "PpiVincent_eff_z": self.eff_z_optimal,
            "PpiVincent_eff_y": self.eff_y_optimal,
            "PpiInternal_eff_z": self.eff_z_ppi_internal,
            "PpiInternal_eff_y": self.eff_y_ppi_internal,

            "success": success,
            "gap_percent": gap_percent,
            "relative_to_ppi_percent": relative_to_ppi,
            "improvement_percent": max(relative_to_ppi, 0.0),

            "total_time": total_time,
            "colony_size": colony_size,
            "total_abandoned": total_abandoned,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,

            "history": self.history.copy(),
            "history_records": self.history_records.copy(),
            "scout_history": self.scout_history.copy(),

            "order": self.order.copy(),
            "enforce_cadsd_order": self.enforce_cadsd_order,
            "same_pi_multiset_check": self.same_pi_multiset_check,

            "validation_mode": self.validation_mode,
            "strict_check_count": self.strict_check_count,

            "vincent_ppi_diag_error": self.vincent_ppi_diag_error,
            "vincent_ppi_projection_error": self.vincent_ppi_projection_error,
            "internal_ppi_diag_error": self.internal_ppi_diag_error,
        }

    def print_results(self, results: dict):
        """Print compact final results."""
        print("=" * 70)
        print(f"FINAL RESULTS (ABC): {results['case']}")
        print("=" * 70)

        print(
            f"Best ABC:     eff_z={results['best_eff_z']:.6f}, "
            f"eff_y={results['best_eff_y']:.6f}"
        )

        print(
            f"Reference Pπ: eff_z={results['optimal_eff_z']:.6f}, "
            f"eff_y={results['optimal_eff_y']:.6f}"
        )

        print(f"ABC vs Pπ:    {results['relative_to_ppi_percent']:+.2f}%")

        print(
            f"Time: {results['total_time']:.2f}s | "
            f"valid/eval: {results['valid_count']}/{results['eval_count']} | "
            f"scouts: {results['total_abandoned']}"
        )

        if len(results["history"]) > 0 and results["history"][0] > 0:
            conv = 100.0 * (results["history"][-1] / results["history"][0] - 1.0)
            print(f"Convergence gain: {conv:+.2f}%")

        print("=" * 70)


class RandomSearchAlgorithm(ABCAlgorithm):
    """
    Pure random-search baseline using the same CaDsd parameterization
    and the same efficiency evaluation as ABC.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.center_evaluated = False
        self.random_history_records = []

    def step(self, n_evals: int = 1, include_center_once: bool = False) -> dict:
        """Run n random candidate evaluations and update the random-search best."""
        start_time = time.time()
        n_evals = int(max(0, n_evals))
        done = 0

        if include_center_once and (not self.center_evaluated) and n_evals > 0:
            center_omega = 0.5 * np.ones((self.M, self.N))
            center_rho = 0.5 * np.ones((self.M, self.N - 1))
            self._food_from_arrays(center_omega, center_rho)
            self.center_evaluated = True
            done += 1

        while done < n_evals:
            omega, rho = self._random_candidate()
            self._food_from_arrays(omega, rho)
            done += 1

        self.history.append(self.global_best_eff_z)

        record = {
            "step_evals": n_evals,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,
            "strict_check_count": self.strict_check_count,
            "elapsed_step": time.time() - start_time,
        }

        self.random_history_records.append(record)

        return record

    def optimize(
        self,
        max_evaluations: int = 1000,
        verbose: bool = True,
        progress_interval: Optional[int] = None,
        include_center_once: bool = True,
    ) -> dict:
        """Standalone random search, useful for quick debugging."""
        start_time = time.time()

        if progress_interval is None:
            progress_interval = max(1, max_evaluations // 20)

        if verbose:
            print("=" * 80)
            print(f"RANDOM SEARCH: {self.case_name}")
            print("=" * 80)
            print(f"Reference Pπ eff_z = {self.eff_z_optimal:.4f}")
            print(
                f"{'eval':>8} | {'Rand_z':>10} | {'Rand_y':>10} | "
                f"{'Rand/Pπ':>8} | {'valid/eval':>12} | {'elapsed':>8}"
            )
            print("-" * 78)

        done = 0

        while done < max_evaluations:
            batch = min(progress_interval, max_evaluations - done)

            self.step(
                batch,
                include_center_once=(include_center_once and done == 0),
            )

            done += batch

            if verbose:
                rand_ratio = (
                    self.global_best_eff_z / self.eff_z_optimal
                    if self.eff_z_optimal > 0 and self.global_best_eff_z > 0
                    else np.nan
                )

                print(
                    f"{done:8d} | "
                    f"{self.global_best_eff_z:10.4f} | "
                    f"{self.global_best_eff_y:10.4f} | "
                    f"{rand_ratio:8.3f} | "
                    f"{self.valid_count}/{self.eval_count:>5} | "
                    f"{time.time() - start_time:7.1f}s"
                )

        results = self._prepare_results(
            time.time() - start_time,
            colony_size=0,
            total_abandoned=0,
        )

        results["search_type"] = "random"
        results["max_evaluations"] = max_evaluations

        return results


print(
    "Corrected ABCAlgorithm and RandomSearchAlgorithm loaded. "
    "Reference = true Vincent Pπ in z/pi order."
)

Corrected ABCAlgorithm and RandomSearchAlgorithm loaded. Reference = true Vincent Pπ in z/pi order.


In [9]:
# ============================================================
# Terminal progress patch: positive ratios only
# ============================================================

def _valid_percent(valid, total):
    return 100.0 * valid / total if total and total > 0 else 0.0


def _ratio_to_ref(value, ref):
    """
    Positive efficiency ratio.
    value/ref = 1.00 means equal to Pπ.
    value/ref = 0.95 means 95% of Pπ.
    value/ref = 1.05 means 5% better than Pπ.
    """
    return value / ref if ref and ref > 0 and value > 0 else np.nan


def _abc_optimize_progress_terminal(
    self,
    colony_size=40,
    max_iterations=100,
    limit=25,
    verbose=True,
    progress_interval=None,
    local_search_interval=25,
    local_search_attempts=2,
    onlooker_factor=0.60,
    early_stopping=True,
    patience=None,
    min_iterations=None,
    rel_tol=1e-5,
    time_limit_seconds=None,
    random_searcher=None,
    random_evals_per_iteration=None,
    random_start_evals=None,
):
    start_time = time.time()

    if progress_interval is None:
        progress_interval = max(1, max_iterations // 20)
    if patience is None:
        patience = max(25, max_iterations // 4)
    if min_iterations is None:
        min_iterations = max(20, max_iterations // 3)

    use_random = random_searcher is not None

    if use_random:
        if random_evals_per_iteration is None:
            random_evals_per_iteration = max(1, int(round(colony_size * (1.0 + onlooker_factor))))
        if random_start_evals is None:
            random_start_evals = colony_size

    if verbose:
        print("=" * 80, flush=True)
        print(f" ABC ALGORITHM - {self.case_name}", flush=True)
        print("=" * 80, flush=True)
        print(" Configuration:", flush=True)
        print(f"   colony_size          = {colony_size}", flush=True)
        print(f"   max_iterations       = {max_iterations}", flush=True)
        print(f"   abandonment limit    = {limit}", flush=True)
        print(f"   objective            = {self.objective}", flush=True)
        print(f"   validation mode      = {getattr(self, 'validation_mode', 'unknown')}", flush=True)
        print(f"   onlooker factor      = {onlooker_factor}", flush=True)
        print(f"   early stopping       = {early_stopping}, patience={patience}", flush=True)
        print(f"   time limit           = {time_limit_seconds}", flush=True)
        print(f"   random search        = {'ON' if use_random else 'OFF'}", flush=True)

        if use_random:
            print(f"   random start evals   = {random_start_evals}", flush=True)
            print(f"   random evals/iter    = {random_evals_per_iteration}", flush=True)

        print(f"   progress interval    = every {progress_interval} iterations", flush=True)
        print(f"   local search interval= every {local_search_interval} iterations", flush=True)
        print(" Reference Pπ efficiency versus SRS:", flush=True)
        print(f"   Pπ_z = {self.eff_z_optimal:.2f}", flush=True)
        print(f"   Pπ_y = {self.eff_y_optimal:.2f}\n", flush=True)

    population = self.initialize_population(colony_size, verbose)

    if len(population) == 0:
        if verbose:
            print("\nFAILED: could not initialize any valid solution.", flush=True)
        return self._prepare_results(time.time() - start_time, colony_size, 0)

    if use_random and random_start_evals and random_start_evals > 0:
        random_searcher.step(random_start_evals, include_center_once=True)

    total_abandoned = 0
    last_improvement_iter = 0
    stopped_early = False
    stopped_by_time = False
    completed_iterations = 0

    if verbose:
        if use_random:
            header = (
                f"{'iter':>5} | "
                f"{'ABC_z/Pπ_z':>10} | {'ABC_y/Pπ_y':>10} | "
                f"{'Rand_z/Pπ_z':>11} | {'Rand_y/Pπ_y':>11} | "
                f"{'scout':>5} | {'ABC val%':>8} | {'Rnd val%':>8} | {'elapsed':>8}"
            )
        else:
            header = (
                f"{'iter':>5} | "
                f"{'ABC_z/Pπ_z':>10} | {'ABC_y/Pπ_y':>10} | "
                f"{'scout':>5} | {'ABC val%':>8} | {'elapsed':>8}"
            )

        print(header, flush=True)
        print("-" * len(header), flush=True)

    for iteration in range(max_iterations):
        progress = (iteration + 1) / max_iterations
        old_best = self.global_best_score

        population = self.employed_bee_phase(population, progress)
        population = self.onlooker_bee_phase(
            population,
            progress,
            onlooker_factor=onlooker_factor,
        )
        population, n_abandoned = self.scout_bee_phase(population, limit, progress)
        total_abandoned += n_abandoned

        if local_search_interval and (iteration + 1) % local_search_interval == 0:
            population = self.local_search_phase(
                population,
                progress,
                attempts=local_search_attempts,
            )

        population = self._inject_elite(population)

        if use_random and random_evals_per_iteration and random_evals_per_iteration > 0:
            random_searcher.step(random_evals_per_iteration, include_center_once=False)

        completed_iterations = iteration + 1

        if self.global_best_score > old_best * (1.0 + rel_tol):
            last_improvement_iter = iteration + 1

        record = {
            "iteration": iteration + 1,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "abc_z_over_ppi_z": _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal),
            "abc_y_over_ppi_y": _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal),
            "n_abandoned": n_abandoned,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,
            "abc_valid_percent": _valid_percent(self.valid_count, self.eval_count),
            "strict_check_count": getattr(self, "strict_check_count", 0),
            "elapsed": time.time() - start_time,
        }

        if use_random:
            record.update({
                "random_best_eff_z": random_searcher.global_best_eff_z,
                "random_best_eff_y": random_searcher.global_best_eff_y,
                "random_best_score": random_searcher.global_best_score,
                "random_z_over_ppi_z": _ratio_to_ref(random_searcher.global_best_eff_z, self.eff_z_optimal),
                "random_y_over_ppi_y": _ratio_to_ref(random_searcher.global_best_eff_y, self.eff_y_optimal),
                "random_eval_count": random_searcher.eval_count,
                "random_valid_count": random_searcher.valid_count,
                "random_valid_percent": _valid_percent(random_searcher.valid_count, random_searcher.eval_count),
            })

        self.history.append(self.global_best_eff_z)
        self.history_records.append(record)
        self.scout_history.append(n_abandoned)

        should_print = (
            verbose
            and (
                iteration == 0
                or (iteration + 1) % progress_interval == 0
                or (iteration + 1) == max_iterations
            )
        )

        if should_print:
            abc_z_ratio = _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal)
            abc_y_ratio = _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal)
            abc_valid = _valid_percent(self.valid_count, self.eval_count)
            elapsed = time.time() - start_time

            if use_random:
                rand_z_ratio = _ratio_to_ref(random_searcher.global_best_eff_z, self.eff_z_optimal)
                rand_y_ratio = _ratio_to_ref(random_searcher.global_best_eff_y, self.eff_y_optimal)
                rand_valid = _valid_percent(random_searcher.valid_count, random_searcher.eval_count)

                print(
                    f"{iteration+1:5d} | "
                    f"{abc_z_ratio:10.2f} | {abc_y_ratio:10.2f} | "
                    f"{rand_z_ratio:11.2f} | {rand_y_ratio:11.2f} | "
                    f"{n_abandoned:5d} | {abc_valid:7.2f}% | {rand_valid:7.2f}% | "
                    f"{elapsed:7.2f}s",
                    flush=True,
                )
            else:
                print(
                    f"{iteration+1:5d} | "
                    f"{abc_z_ratio:10.2f} | {abc_y_ratio:10.2f} | "
                    f"{n_abandoned:5d} | {abc_valid:7.2f}% | {elapsed:7.2f}s",
                    flush=True,
                )

        if (
            early_stopping
            and (iteration + 1) >= min_iterations
            and (iteration + 1 - last_improvement_iter) >= patience
        ):
            stopped_early = True
            if verbose:
                print(
                    f"    Early stop at iter {iteration+1}: "
                    f"no relative improvement > {rel_tol:g} for {patience} iterations.",
                    flush=True,
                )
            break

        if (
            time_limit_seconds is not None
            and (iteration + 1) >= min_iterations
            and (time.time() - start_time) >= float(time_limit_seconds)
        ):
            stopped_by_time = True
            if verbose:
                print(f"    Time stop at iter {iteration+1}: reached {time_limit_seconds}s budget.", flush=True)
            break

    total_time = time.time() - start_time
    results = self._prepare_results(total_time, colony_size, total_abandoned)

    results["abc_z_over_Ppi_z"] = _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal)
    results["abc_y_over_Ppi_y"] = _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal)
    results["abc_valid_percent"] = _valid_percent(self.valid_count, self.eval_count)

    if use_random:
        results.update({
            "random_best_eff_z": random_searcher.global_best_eff_z,
            "random_best_eff_y": random_searcher.global_best_eff_y,
            "random_best_score": random_searcher.global_best_score,
            "random_z_over_Ppi_z": _ratio_to_ref(random_searcher.global_best_eff_z, self.eff_z_optimal),
            "random_y_over_Ppi_y": _ratio_to_ref(random_searcher.global_best_eff_y, self.eff_y_optimal),
            "random_eval_count": random_searcher.eval_count,
            "random_valid_count": random_searcher.valid_count,
            "random_valid_percent": _valid_percent(random_searcher.valid_count, random_searcher.eval_count),
            "random_history": random_searcher.history.copy(),
            "random_history_records": getattr(random_searcher, "random_history_records", []).copy(),
            "random_search_enabled": True,
            "random_evals_per_iteration": random_evals_per_iteration,
            "random_start_evals": random_start_evals,
        })
    else:
        results["random_search_enabled"] = False

    results["stopped_early"] = stopped_early
    results["stopped_by_time"] = stopped_by_time
    results["completed_iterations"] = completed_iterations
    results["strict_check_count"] = getattr(self, "strict_check_count", 0)

    return results


ABCAlgorithm.optimize = _abc_optimize_progress_terminal

## MU284 Dataset

In [80]:
import pandas as pd
import numpy as np
from pathlib import Path

# Try the normal repository path first, then local filenames.
MU284_PATH_CANDIDATES = [
    Path("/home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv"),
    Path("MU284.csv"),
    Path("MU284_filtered.csv"),
    Path("/mnt/data/MU284.csv"),
    Path("/mnt/data/MU284_filtered.csv"),
]

MU284_PATH = None
for candidate in MU284_PATH_CANDIDATES:
    if candidate.exists():
        MU284_PATH = candidate
        break

if MU284_PATH is None:
    raise FileNotFoundError(
        "Could not find MU284 data. Put MU284.csv or MU284_filtered.csv in the notebook folder, "
        "or edit MU284_PATH_CANDIDATES."
    )

# # df_full = pd.read_csv(MU284_PATH)
# df = df_full.sample(n=30, random_state=500).reset_index(drop=True)
df = pd.read_csv("synthetic_population_N30_y_z_pi.csv")
print("=" * 80)
print("MU284 DATA LOADED")
print("=" * 80)
print(f"path = {MU284_PATH}")
print(f"N = {len(df)}")
print(f"columns = {list(df.columns)}")
df['CONST'] = 1
y_var = "P85"

print("\nCorrelations with P85:")
for col in ['P85', 'P75', 'RMT85', 'CS82', 'SS82', 'S82', 'ME84', 'REV84', 'REG', 'CL']:
    if col in df.columns and y_var in df.columns:
        corr = np.corrcoef(df[y_var].to_numpy(dtype=float), df[col].to_numpy(dtype=float))[0, 1]
        print(f"Corr({y_var}, {col}) = {corr:.3f}")

MU284 DATA LOADED
path = /home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv
N = 30
columns = ['id', 'y', 'z70', 'z80', 'z90', 'x75', 'x90', 'equal']

Correlations with P85:


In [81]:
# ============================================================
# CELL: Experiment settings
# ============================================================
# Edit only this cell to change the experiment.
#
# This cell defines:
#   - auxiliary variables z,
#   - variables used to construct inclusion probabilities,
#   - sample sizes,
#   - ABC/random-search seeds,
#   - ABC tuning parameters,
#   - output CSV file name.
#
# Place this cell after loading the data and before the main run.
# ============================================================

y_var = "y"

test_cases = [
    ("z70", "x75", "Corr z=0.70, pi=0.75"),
    ("z80", "x75", "Corr z=0.80, pi=0.75"),
    ("z90", "x75", "Corr z=0.90, pi=0.75"),

    ("z70", "x90", "Corr z=0.70, pi=0.90"),
    ("z80", "x90", "Corr z=0.80, pi=0.90"),
    ("z90", "x90", "Corr z=0.90, pi=0.90"),

    ("z70", "equal", "Corr z=0.70, Equal pi"),
    ("z80", "equal", "Corr z=0.80, Equal pi"),
    ("z90", "equal", "Corr z=0.90, Equal pi"),
]
# df has N = 30 in the current test, so n must be <= 30.
sample_sizes = [5]

# y_var = "P85"

ABC_RANDOM_SEED = 12345
RANDOM_SEARCH_SEED = 54321

# Options: "eff_z", "eff_y", "harmonic", "mean"
OBJECTIVE = "eff_z"

SIM_PARAMS = {
    "colony_size": 20,
    "max_iterations": 1000,
    "limit": 25,
    "onlooker_factor": 0.55,
    "local_search_attempts": 1,
    "local_search_interval": 12,
    "progress_interval": 50,
    "validation_mode": "fast",
    "time_limit_seconds": None,

    "early_stopping": False,
    "patience": 50,
    "min_iterations": 20,
    "rel_tol": 1e-5,

    "random_start_evals": 20,
    "random_evals_per_iteration": 31,
}

OUTPUT_CSV = "abc_random_mu284_corrected_vincent_reference.csv"

print("EXPERIMENT SETTINGS LOADED")
print(f"test_cases = {test_cases}")
print(f"sample_sizes = {sample_sizes}")
print(f"y_var = {y_var}")
print(f"OBJECTIVE = {OBJECTIVE}")
print(f"OUTPUT_CSV = {OUTPUT_CSV}")

EXPERIMENT SETTINGS LOADED
test_cases = [('z70', 'x75', 'Corr z=0.70, pi=0.75'), ('z80', 'x75', 'Corr z=0.80, pi=0.75'), ('z90', 'x75', 'Corr z=0.90, pi=0.75'), ('z70', 'x90', 'Corr z=0.70, pi=0.90'), ('z80', 'x90', 'Corr z=0.80, pi=0.90'), ('z90', 'x90', 'Corr z=0.90, pi=0.90'), ('z70', 'equal', 'Corr z=0.70, Equal pi'), ('z80', 'equal', 'Corr z=0.80, Equal pi'), ('z90', 'equal', 'Corr z=0.90, Equal pi')]
sample_sizes = [5]
y_var = y
OBJECTIVE = eff_z
OUTPUT_CSV = abc_random_mu284_corrected_vincent_reference.csv


In [82]:
# ============================================================
# RUN PRECHECK BEFORE MAIN RUN
# ============================================================
# This cell uses the helper functions already defined earlier:
#   - ppi_efficiency_for_order()
#   - precheck_one_case()
# ============================================================

RUN_PRECHECK = True

if RUN_PRECHECK:
    print("=" * 80)
    print("PRECHECK BEFORE MAIN RUN")
    print("=" * 80)

    precheck_rows = []

    for z_name, x_name, label in test_cases:
        for n in sample_sizes:
            precheck_rows.append(
                precheck_one_case(
                    df=df,
                    y_name=y_var,
                    z_name=z_name,
                    x_name=x_name,
                    n=n,
                )
            )

    precheck_df = pd.DataFrame(precheck_rows)

    display_cols = [
        "z",
        "x_for_pi",
        "n",
        "sum_pi",
        "ABC_matches_Vincent_z",
        "same_pi_multiset",
        "CaDsd_same_pi",
        "Ppi_eff_z_direct",
        "Ppi_eff_z_probe",
        "Ppi_diag_error",
        "CaDsd_diag_error",
    ]

    display(precheck_df[display_cols].round(6))

    if not precheck_df["ABC_matches_Vincent_z"].all():
        raise RuntimeError("PRECHECK FAILED: ABC reference does not match Vincent Pπ.")

    if not precheck_df["same_pi_multiset"].all():
        raise RuntimeError("PRECHECK FAILED: pi values changed after internal ordering.")

    if not precheck_df["CaDsd_same_pi"].all():
        raise RuntimeError("PRECHECK FAILED: CaDsd does not use the same pi values.")

    if precheck_df["Ppi_diag_error"].max() > 1e-6:
        raise RuntimeError("PRECHECK FAILED: Pπ diagonal error is too large.")

    print("PRECHECK PASSED. You can now run the main cell.")

else:
    print("RUN_PRECHECK=False: skipped precheck.")

PRECHECK BEFORE MAIN RUN


,z,x_for_pi,n,sum_pi,ABC_matches_Vincent_z,same_pi_multiset,CaDsd_same_pi,Ppi_eff_z_direct,Ppi_eff_z_probe,Ppi_diag_error,CaDsd_diag_error
0,z70,x75,5,5.0,True,True,True,6.316126,6.316126,0.0,0.000008
1,z80,x75,5,5.0,True,True,True,5.862891,5.862891,0.0,0.000008
2,z90,x75,5,5.0,True,True,True,11.804756,11.804756,0.0,0.000008
3,z70,x90,5,5.0,True,True,True,15.689931,15.689931,0.0,0.000004
4,z80,x90,5,5.0,True,True,True,10.422058,10.422058,0.0,0.000004
5,z90,x90,5,5.0,True,True,True,21.133730,21.133730,0.0,0.000004
6,z70,equal,5,5.0,True,True,True,8.542162,8.542162,0.0,0.000013
7,z80,equal,5,5.0,True,True,True,7.283316,7.283316,0.0,0.000013
8,z90,equal,5,5.0,True,True,True,10.021973,10.021973,0.0,0.000013


PRECHECK PASSED. You can now run the main cell.


In [83]:
# ============================================================
# MAIN RUN: ABC + Random Search
# Reference in ratios = corrected Vincent ordered Pπ
# ============================================================

RUN_MAIN = True

if RUN_MAIN:
    print("=" * 80)
    print("ABC + RANDOM SEARCH ON MU284 DATA")
    print("Reference: corrected Vincent z/pi ordered Pπ")
    print("=" * 80)

    N = len(df)
    all_results = []
    abc_run_details = {}

    for z_name, x_name, corr_label in test_cases:
        y_raw = df[y_var].to_numpy(dtype=float)
        z_raw = df[z_name].to_numpy(dtype=float)
        x_raw = df[x_name].to_numpy(dtype=float)

        actual_corr = np.corrcoef(y_raw, z_raw)[0, 1]

        print("\n" + "=" * 80)
        print(f"{corr_label}: z = {z_name}; π from {x_name}; Corr({y_var}, {z_name}) = {actual_corr:.3f}")
        print("=" * 80)

        for n in sample_sizes:
            print(f"\nRunning case: z={z_name}, x_for_pi={x_name}, N={N}, n={n}")

            # ------------------------------------------------
            # 1) Inclusion probabilities
            # ------------------------------------------------
            pik_raw = inclusionprobabilities(x_raw, n)

            if not np.isclose(pik_raw.sum(), n, atol=1e-8):
                raise RuntimeError(
                    f"sum(pi) = {pik_raw.sum()}, but expected n = {n}."
                )

            if np.any(pik_raw <= 0) or np.any(pik_raw >= 1):
                raise RuntimeError(
                    "This run requires 0 < pi_i < 1 for Ppi/CaDsd."
                )

            # ------------------------------------------------
            # 2) Vincent order: rank by z/pi
            # ------------------------------------------------
            z_over_pi = z_raw / pik_raw
            sort_idx = np.argsort(z_over_pi)

            y_sorted = y_raw[sort_idx]
            z_sorted = z_raw[sort_idx]
            pik_sorted = pik_raw[sort_idx]

            # ------------------------------------------------
            # 3) SRS variances
            # ------------------------------------------------
            var_srs_y = N**2 * (1.0 - n / N) * np.var(y_raw, ddof=1) / n
            var_srs_z = N**2 * (1.0 - n / N) * np.var(z_raw, ddof=1) / n

            # ------------------------------------------------
            # 4) Build ABC and random-search objects
            # ------------------------------------------------
            run_key = f"{z_name}_{x_name}_N{N}_n{n}"
            seed_shift = 1000 * n + len(all_results)

            abc = ABCAlgorithm(
                y_sorted=y_sorted,
                z_sorted=z_sorted,
                pik_sorted=pik_sorted,
                var_srs_y=var_srs_y,
                var_srs_z=var_srs_z,
                M=n,
                n=n,
                case_name=run_key,
                objective=OBJECTIVE,
                enforce_cadsd_order=True,
                random_state=ABC_RANDOM_SEED + seed_shift,
                validation_mode=SIM_PARAMS["validation_mode"],
                eigen_check_interval=0,
                initial_strict_checks=2,
            )

            random_search = RandomSearchAlgorithm(
                y_sorted=y_sorted,
                z_sorted=z_sorted,
                pik_sorted=pik_sorted,
                var_srs_y=var_srs_y,
                var_srs_z=var_srs_z,
                M=n,
                n=n,
                case_name=run_key + "_random",
                objective=OBJECTIVE,
                enforce_cadsd_order=True,
                random_state=RANDOM_SEARCH_SEED + seed_shift,
                validation_mode=SIM_PARAMS["validation_mode"],
                eigen_check_interval=0,
                initial_strict_checks=2,
            )

            same_pi_check = np.allclose(
                np.sort(abc.input_pi),
                np.sort(abc.pik_sorted),
                atol=1e-12,
                rtol=1e-12,
            )

            if not same_pi_check:
                raise RuntimeError(f"Same-π check failed for {run_key}.")

            # ------------------------------------------------
            # 5) Run ABC with random-search baseline
            # ------------------------------------------------
            res = abc.optimize(
                colony_size=SIM_PARAMS["colony_size"],
                max_iterations=SIM_PARAMS["max_iterations"],
                limit=SIM_PARAMS["limit"],
                verbose=True,
                progress_interval=SIM_PARAMS["progress_interval"],
                local_search_interval=SIM_PARAMS["local_search_interval"],
                local_search_attempts=SIM_PARAMS["local_search_attempts"],
                onlooker_factor=SIM_PARAMS["onlooker_factor"],
                early_stopping=SIM_PARAMS["early_stopping"],
                patience=SIM_PARAMS["patience"],
                min_iterations=SIM_PARAMS["min_iterations"],
                rel_tol=SIM_PARAMS["rel_tol"],
                time_limit_seconds=SIM_PARAMS["time_limit_seconds"],
                random_searcher=random_search,
                random_start_evals=SIM_PARAMS["random_start_evals"],
                random_evals_per_iteration=SIM_PARAMS["random_evals_per_iteration"],
            )

            # ------------------------------------------------
            # 6) Final ratios relative to corrected Vincent Pπ
            # ------------------------------------------------
            ppi_z = res["optimal_eff_z"]
            ppi_y = res["optimal_eff_y"]

            abc_z_ratio = _ratio_to_ref(res["best_eff_z"], ppi_z)
            abc_y_ratio = _ratio_to_ref(res["best_eff_y"], ppi_y)
            random_z_ratio = _ratio_to_ref(res["random_best_eff_z"], ppi_z)
            random_y_ratio = _ratio_to_ref(res["random_best_eff_y"], ppi_y)

            abc_valid_pct = _valid_percent(res["valid_count"], res["eval_count"])
            random_valid_pct = _valid_percent(
                res["random_valid_count"],
                res["random_eval_count"],
            )

            # ------------------------------------------------
            # 7) Compact result print
            # ------------------------------------------------
            print(
                f"Result | "
                f"Pπ_z={ppi_z:.2f}, "
                f"ABC_z/Pπ_z={abc_z_ratio:.3f}, "
                f"Random_z/Pπ_z={random_z_ratio:.3f}, "
                f"ABC valid={abc_valid_pct:.1f}%, "
                f"time={res['total_time']:.1f}s"
            )

            # ------------------------------------------------
            # 8) Save row
            # ------------------------------------------------
            all_results.append({
                "z": z_name,
                "x_for_pi": x_name,
                "corr": actual_corr,
                "n": n,
                "N": N,

                "Ppi_z_eff_vs_SRS": ppi_z,
                "Ppi_y_eff_vs_SRS": ppi_y,

                "ABC_z_eff_vs_SRS": res["best_eff_z"],
                "ABC_y_eff_vs_SRS": res["best_eff_y"],
                "Random_z_eff_vs_SRS": res["random_best_eff_z"],
                "Random_y_eff_vs_SRS": res["random_best_eff_y"],

                "ABC_z_over_Ppi_z": abc_z_ratio,
                "ABC_y_over_Ppi_y": abc_y_ratio,
                "Random_z_over_Ppi_z": random_z_ratio,
                "Random_y_over_Ppi_y": random_y_ratio,

                "same_pi_check": same_pi_check,
                "ABC_valid_percent": abc_valid_pct,
                "Random_valid_percent": random_valid_pct,

                "time_seconds": res["total_time"],
                "completed_iterations": res.get(
                    "completed_iterations",
                    SIM_PARAMS["max_iterations"],
                ),
                "stopped_early": res.get("stopped_early", False),
                "stopped_by_time": res.get("stopped_by_time", False),
                "abc_eval_count": res["eval_count"],
                "random_eval_count": res["random_eval_count"],
            })

            abc_run_details[run_key] = {
                "abc": abc,
                "random_search": random_search,
                "result": res,
                "sort_idx": sort_idx,
                "z_name": z_name,
                "x_name": x_name,
                "n": n,
                "N": N,
            }

            # Save after every finished case.
            pd.DataFrame(all_results).to_csv(OUTPUT_CSV, index=False)
            print(f"Saved partial results to: {OUTPUT_CSV}")

    # --------------------------------------------------------
    # 9) Final summary
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("FINAL SUMMARY TABLE")
    print("=" * 80)

    df_res = pd.DataFrame(all_results)

    summary_cols = [
        "z",
        "x_for_pi",
        "corr",
        "n",
        "N",
        "Ppi_z_eff_vs_SRS",
        "ABC_z_over_Ppi_z",
        "Random_z_over_Ppi_z",
        "ABC_y_over_Ppi_y",
        "Random_y_over_Ppi_y",
        "ABC_valid_percent",
        "Random_valid_percent",
        "time_seconds",
        "completed_iterations",
        "same_pi_check",
    ]

    df_print = df_res[summary_cols].copy()
    num_cols = df_print.select_dtypes(include=[np.number]).columns
    df_print[num_cols] = df_print[num_cols].round(3)

    display(df_print)
    print(df_print.to_string(index=False))

    df_res.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved final results to: {OUTPUT_CSV}")
    print("Complete run objects are available in: abc_run_details")

else:
    print("RUN_MAIN=False: skipped main ABC run.")

ABC + RANDOM SEARCH ON MU284 DATA
Reference: corrected Vincent z/pi ordered Pπ

Corr z=0.70, pi=0.75: z = z70; π from x75; Corr(y, z70) = 0.700

Running case: z=z70, x_for_pi=x75, N=30, n=5
 ABC ALGORITHM - z70_x75_N30_n5
 Configuration:
   colony_size          = 20
   max_iterations       = 1000
   abandonment limit    = 25
   objective            = eff_z
   validation mode      = fast
   onlooker factor      = 0.55
   early stopping       = False, patience=50
   time limit           = None
   random search        = ON
   random start evals   = 20
   random evals/iter    = 31
   progress interval    = every 50 iterations
   local search interval= every 12 iterations
 Reference Pπ efficiency versus SRS:
   Pπ_z = 6.32


   Pπ_y = 2.96

Initializing 20 food sources...

Initialized 20 food sources.
Best initial: eff_z=1.1965, eff_y=2.4345
 iter | ABC_z/Pπ_z | ABC_y/Pπ_y | Rand_z/Pπ_z | Rand_y/Pπ_y | scout | ABC val% | Rnd val% |  elapsed
----------------------------------------------------------------------------------------------------
    1 |       0.19 |       0.83 |        0.19 |        0.70 |     0 |  100.00% |  100.00% |    0.51s
   50 |       0.31 |       0.92 |        0.21 |        0.67 |     0 |   98.74% |   99.87% |   16.59s
  100 |       0.36 |       1.13 |        0.21 |        0.67 |     1 |   98.12% |   99.87% |   35.85s
  150 |       0.38 |       1.15 |        0.21 |        0.67 |     0 |   97.87% |   99.89% |   55.03s
  200 |       0.40 |       1.01 |        0.22 |        0.91 |     0 |   98.18% |   99.86% |   74.67s
  250 |       0.42 |       1.06 |        0.22 |        0.91 |     0 |   98.53% |   99.85% |   93.85s
  300 |       0.44 |       1.19 |        0.22 |        0.91 |     0 |   9

,z,x_for_pi,corr,n,N,Ppi_z_eff_vs_SRS,ABC_z_over_Ppi_z,Random_z_over_Ppi_z,ABC_y_over_Ppi_y,Random_y_over_Ppi_y,ABC_valid_percent,Random_valid_percent,time_seconds,completed_iterations,same_pi_check
0,z70,x75,0.7,5,30,6.316,0.549,0.223,1.190,0.907,93.904,99.819,394.821,1000,True
1,z80,x75,0.8,5,30,5.863,0.422,0.241,1.033,0.783,99.723,99.810,396.398,1000,True
2,z90,x75,0.9,5,30,11.805,0.504,0.162,0.699,0.539,98.042,99.823,405.116,1000,True
3,z70,x90,0.7,5,30,15.690,0.314,0.127,1.160,0.898,98.116,99.874,398.064,1000,True
4,z80,x90,0.8,5,30,10.422,0.322,0.192,1.444,0.847,92.426,99.819,416.642,1000,True
5,z90,x90,0.9,5,30,21.134,0.361,0.149,1.098,0.897,98.106,99.826,408.655,1000,True
6,z70,equal,0.7,5,30,8.542,1.354,0.364,0.927,0.687,91.201,99.687,403.454,1000,True
7,z80,equal,0.8,5,30,7.283,1.434,0.394,0.993,0.741,90.840,99.636,413.020,1000,True
8,z90,equal,0.9,5,30,10.022,2.139,0.346,1.185,0.589,90.135,99.674,411.947,1000,True


  z x_for_pi  corr  n  N  Ppi_z_eff_vs_SRS  ABC_z_over_Ppi_z  Random_z_over_Ppi_z  ABC_y_over_Ppi_y  Random_y_over_Ppi_y  ABC_valid_percent  Random_valid_percent  time_seconds  completed_iterations  same_pi_check
z70      x75   0.7  5 30             6.316             0.549                0.223             1.190                0.907             93.904                99.819       394.821                  1000           True
z80      x75   0.8  5 30             5.863             0.422                0.241             1.033                0.783             99.723                99.810       396.398                  1000           True
z90      x75   0.9  5 30            11.805             0.504                0.162             0.699                0.539             98.042                99.823       405.116                  1000           True
z70      x90   0.7  5 30            15.690             0.314                0.127             1.160                0.898             98.116         

In [84]:
# ============================================================
# FINAL DESIGN DIAGNOSTIC CHECK
# ============================================================
# This checks the final best ABC kernels.
#
# It reports both:
#   1. strict numerical validity
#   2. practical numerical validity
#
# It does NOT stop the notebook unless you set RAISE_ON_FAIL=True.
# ============================================================

RUN_FINAL_DESIGN_CHECK = True
RAISE_ON_FAIL = False

STRICT_TOL = 1e-3
PRACTICAL_TOL = 2e-2


def check_final_design_diagnostic(
    abc,
    result,
    strict_tol=STRICT_TOL,
    practical_tol=PRACTICAL_TOL,
):
    if result.get("best_omega") is None or result.get("best_rho") is None:
        raise ValueError(f"No best omega/rho found for {abc.case_name}.")

    K = CaDsd(
        pi=abc.pik_sorted,
        M=abc.M,
        omega=result["best_omega"],
        rho=result["best_rho"],
    )["K"].astype(np.complex128)

    N = abc.N
    n = abc.n

    diag_K = np.real(np.diag(K))
    target_pi_internal = abc.pik_sorted_desc

    hermitian_error = float(np.max(np.abs(K - K.conj().T)))
    projection_error = float(np.max(np.abs(K @ K - K)))

    trace_K = float(np.real(np.trace(K)))
    trace_error = abs(trace_K - n)

    eigvals = np.linalg.eigvalsh(K)
    eig_min = float(np.min(eigvals))
    eig_max = float(np.max(eigvals))
    rank_est = int(np.sum(eigvals > 0.5))

    diag_error_ordered = float(np.max(np.abs(diag_K - target_pi_internal)))
    diag_error_multiset = float(
        np.max(np.abs(np.sort(diag_K) - np.sort(abc.input_pi)))
    )

    # For DSD:
    # pi_ij = pi_i pi_j - |K_ij|^2, i != j
    pi2 = np.outer(diag_K, diag_K) - np.abs(K) ** 2
    off_diag = ~np.eye(N, dtype=bool)

    min_second_order_pi = float(np.min(pi2[off_diag]))
    max_second_order_pi = float(np.max(pi2[off_diag]))

    strict_ok = (
        hermitian_error <= strict_tol
        and projection_error <= strict_tol
        and trace_error <= strict_tol
        and rank_est == n
        and eig_min >= -strict_tol
        and eig_max <= 1.0 + strict_tol
        and diag_error_ordered <= strict_tol
        and diag_error_multiset <= strict_tol
        and min_second_order_pi >= -strict_tol
    )

    practical_ok = (
        hermitian_error <= practical_tol
        and projection_error <= practical_tol
        and trace_error <= practical_tol
        and rank_est == n
        and eig_min >= -practical_tol
        and eig_max <= 1.0 + practical_tol
        and diag_error_ordered <= practical_tol
        and diag_error_multiset <= practical_tol
        and min_second_order_pi >= -practical_tol
    )

    return {
        "case": abc.case_name,
        "strict_ok": strict_ok,
        "practical_ok": practical_ok,

        "trace_K": trace_K,
        "trace_error": trace_error,
        "rank_est": rank_est,

        "hermitian_error": hermitian_error,
        "projection_error": projection_error,

        "diag_error_ordered": diag_error_ordered,
        "diag_error_multiset": diag_error_multiset,

        "eig_min": eig_min,
        "eig_max": eig_max,

        "min_second_order_pi": min_second_order_pi,
        "max_second_order_pi": max_second_order_pi,

        "ABC_z_over_Ppi_z": (
            result["best_eff_z"] / result["optimal_eff_z"]
            if result["optimal_eff_z"] > 0
            else np.nan
        ),
        "ABC_y_over_Ppi_y": (
            result["best_eff_y"] / result["optimal_eff_y"]
            if result["optimal_eff_y"] > 0
            else np.nan
        ),
    }


if RUN_FINAL_DESIGN_CHECK:
    if "abc_run_details" not in globals() or len(abc_run_details) == 0:
        raise RuntimeError("No ABC run objects found. Run the main ABC cell first.")

    rows = []

    for run_key, item in abc_run_details.items():
        rows.append(
            check_final_design_diagnostic(
                abc=item["abc"],
                result=item["result"],
            )
        )

    final_design_check_df = pd.DataFrame(rows)

    display_cols = [
        "case",
        "strict_ok",
        "practical_ok",
        "trace_K",
        "trace_error",
        "rank_est",
        "projection_error",
        "diag_error_ordered",
        "eig_min",
        "eig_max",
        "min_second_order_pi",
        "ABC_z_over_Ppi_z",
        "ABC_y_over_Ppi_y",
    ]

    out = final_design_check_df[display_cols].copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(8)

    display(out)

    print("=" * 80)
    print("FINAL DESIGN CHECK SUMMARY")
    print("=" * 80)
    print(f"Strict tolerance     = {STRICT_TOL}")
    print(f"Practical tolerance  = {PRACTICAL_TOL}")
    print(f"Strict OK designs    = {final_design_check_df['strict_ok'].sum()} / {len(final_design_check_df)}")
    print(f"Practical OK designs = {final_design_check_df['practical_ok'].sum()} / {len(final_design_check_df)}")

    if final_design_check_df["practical_ok"].all():
        print("\nPractical check passed.")
        print("The final kernels are numerically close to fixed-size, inclusion-probability-preserving DSD kernels.")
    else:
        print("\nSome designs failed even the practical check.")
        bad = final_design_check_df.loc[
            ~final_design_check_df["practical_ok"],
            display_cols,
        ]
        display(bad)

        if RAISE_ON_FAIL:
            raise RuntimeError("FINAL DESIGN PRACTICAL CHECK FAILED.")

else:
    print("RUN_FINAL_DESIGN_CHECK=False: skipped final design check.")

,case,strict_ok,practical_ok,trace_K,trace_error,rank_est,projection_error,diag_error_ordered,eig_min,eig_max,min_second_order_pi,ABC_z_over_Ppi_z,ABC_y_over_Ppi_y
0,z70_x75_N30_n5,False,True,5.006334,0.006334,5,0.000998,0.000992,-3.400000e-07,1.006175,5.800000e-06,0.549030,1.190267
1,z80_x75_N30_n5,False,True,5.005954,0.005954,5,0.000968,0.000961,-3.300000e-07,1.006038,1.131000e-04,0.421899,1.032684
2,z90_x75_N30_n5,False,True,5.005961,0.005961,5,0.001003,0.000997,-3.600000e-07,1.005993,1.797300e-04,0.504478,0.699036
3,z70_x90_N30_n5,False,True,5.006701,0.006701,5,0.001000,0.000994,-3.600000e-07,1.006654,9.441000e-05,0.313709,1.160498
4,z80_x90_N30_n5,False,True,5.011937,0.011937,5,0.000984,0.000978,-3.700000e-07,1.006940,1.726500e-04,0.322418,1.443841
5,z90_x90_N30_n5,False,True,5.006988,0.006988,5,0.001002,0.000995,-3.600000e-07,1.006972,2.270000e-05,0.360994,1.098237
6,z70_equal_N30_n5,False,True,5.011235,0.011235,5,0.000916,0.000910,-3.500000e-07,1.006997,3.500000e-07,1.354084,0.927447
7,z80_equal_N30_n5,False,True,5.013890,0.013890,5,0.000997,0.000986,-3.500000e-07,1.010931,1.000000e-08,1.433803,0.992725
8,z90_equal_N30_n5,False,True,5.013393,0.013392,5,0.000850,0.000843,-3.900000e-07,1.009322,6.100000e-07,2.138562,1.184654


FINAL DESIGN CHECK SUMMARY
Strict tolerance     = 0.001
Practical tolerance  = 0.02
Strict OK designs    = 0 / 9
Practical OK designs = 9 / 9

Practical check passed.
The final kernels are numerically close to fixed-size, inclusion-probability-preserving DSD kernels.


# Vincent's Idea(Reciprocal)

In [13]:
import pandas as pd

# Step 1: Build K_ppi and export for R
df     = pd.read_csv('../populations/store/MU284_filtered.csv')
y_raw  = df['P85'].values
z_raw  = df['ME84'].values
x_raw  = df['P75'].values
N, n   = 281, 5

pik_raw    = inclusionprobabilities(x_raw, n)
sort_idx   = np.argsort(z_raw / pik_raw)     # Vincent z/pi order
pik_sorted = pik_raw[sort_idx]

# Step 1a: K_ppi در Vincent z/pi order
V_ppi    = Ppi(pik_sorted)
K_ppi_1  = np.real(V_ppi @ V_ppi.T)

# Step 1b: reorder به decreasing diagonal 
desc_idx = np.argsort(np.diag(K_ppi_1))[::-1]
K_ppi_2  = K_ppi_1[np.ix_(desc_idx, desc_idx)]
pik_desc = pik_sorted[desc_idx]

print(f"K_ppi_2 diagonal (first 5): {np.diag(K_ppi_2)[:5]}")
print(f"Is diagonal descending: {np.all(np.diff(np.diag(K_ppi_2)) <= 0)}")

# Export
np.savetxt('K_ppi_2_ME84_n5.csv', K_ppi_2, delimiter=',')
np.savetxt('pik_desc_ME84_n5.csv', pik_desc, delimiter=',')
print("Saved: K_ppi_2_ME84_n5.csv and pik_desc_ME84_n5.csv")

# R code for Bardia
print("\n--- R CODE ---")
print("source('Reciprocal_CaDsd')")
print("K <- as.matrix(read.csv('K_ppi_2_ME84_n5.csv', header=FALSE))")
print("result <- Reciprocal_CaDsd(K)")
print("write.csv(result$omega, 'omega_ppi_ME84_n5.csv', row.names=FALSE)")
print("write.csv(result$rho,   'rho_ppi_ME84_n5.csv',   row.names=FALSE)")
print("# Verify:")
print("K_check <- CaDsd(pi=diag(K), M=5, omega=result$omega, rho=result$rho)$K")
print("cat('max diff:', max(abs(K_check - K)))")

K_ppi_2 diagonal (first 5): [0.1012027  0.08726899 0.08653564 0.08653564 0.07920211]
Is diagonal descending: True
Saved: K_ppi_2_ME84_n5.csv and pik_desc_ME84_n5.csv

--- R CODE ---
source('Reciprocal_CaDsd')
K <- as.matrix(read.csv('K_ppi_2_ME84_n5.csv', header=FALSE))
result <- Reciprocal_CaDsd(K)
write.csv(result$omega, 'omega_ppi_ME84_n5.csv', row.names=FALSE)
write.csv(result$rho,   'rho_ppi_ME84_n5.csv',   row.names=FALSE)
# Verify:
K_check <- CaDsd(pi=diag(K), M=5, omega=result$omega, rho=result$rho)$K
cat('max diff:', max(abs(K_check - K)))


In [14]:
# Data prep
df      = pd.read_csv('../populations/store/MU284_filtered.csv')
y_raw   = df['P85'].values
z_raw   = df['ME84'].values
x_raw   = df['P75'].values
N, n    = 281, 5

pik_raw    = inclusionprobabilities(x_raw, n)
sort_idx   = np.argsort(z_raw / pik_raw)
pik_sorted = pik_raw[sort_idx]
y_sorted   = y_raw[sort_idx]
z_sorted   = z_raw[sort_idx]

var_srs_y = N**2 * (1 - n/N) * np.var(y_raw, ddof=1) / n
var_srs_z = N**2 * (1 - n/N) * np.var(z_raw, ddof=1) / n

# Ppi reference
V_ppi      = Ppi(pik_sorted)
K_ppi      = np.real(V_ppi @ V_ppi.T)
I_N        = np.eye(N)
Dpi_inv    = np.diag(1.0 / pik_sorted)
A_ppi      = K_ppi * (I_N - K_ppi)
var_z_ppi  = float(np.real(z_sorted @ Dpi_inv @ A_ppi @ Dpi_inv @ z_sorted))
eff_z_optimal = var_srs_z / var_z_ppi

# Load omega and rho
omega_ppi = pd.read_csv('omega_ppi_ME84_n5.csv').values
rho_ppi   = pd.read_csv('rho_ppi_ME84_n5.csv').values

# Build K from warm start
K_warm  = np.real(CaDsd(pi=pik_sorted, M=n, omega=omega_ppi, rho=rho_ppi)["K"])
A_warm  = K_warm * (I_N - K_warm)
var_z_warm = float(np.real(z_sorted @ Dpi_inv @ A_warm @ Dpi_inv @ z_sorted))
eff_z_warm = var_srs_z / var_z_warm

print(f"eff_z (warm start) = {eff_z_warm:.4f}")
print(f"eff_z (Pπ)         = {eff_z_optimal:.4f}")
print(f"Rz,0 = {eff_z_warm / eff_z_optimal:.4f}")

eff_z (warm start) = 96.9534
eff_z (Pπ)         = 214.5640
Rz,0 = 0.4519


# ABC Warm Start

In [15]:
# چک شرط Vincent
pik_desc = np.sort(pik_raw)[::-1]
z_reorder = z_raw[np.argsort(pik_raw)[::-1]]

z_over_pi = z_reorder / pik_desc
print(f"Is z/pi monotone increasing: {np.all(np.diff(z_over_pi) > 0)}")
print(f"Is z/pi monotone decreasing: {np.all(np.diff(z_over_pi) < 0)}")
print(f"Corr(rank_pi, rank_z_over_pi): {np.corrcoef(np.arange(len(pik_desc)), z_over_pi)[0,1]:.4f}")

Is z/pi monotone increasing: False
Is z/pi monotone decreasing: False
Corr(rank_pi, rank_z_over_pi): -0.4229


In [16]:
# ساخت z مصنوعی طبق روش Vincent
np.random.seed(123)

pik_desc   = np.sort(pik_raw)[::-1]
N          = len(pik_desc)

# z1/pi1 fix
z1_over_pi1 = 5.0 / pik_desc[0]

# ratios صعودی 
ratios = np.cumsum(np.concatenate([[z1_over_pi1],
                                    np.random.uniform(0.2, 1, N-1)]))

# z = ratios * pi
z_synthetic = ratios * pik_desc

# چک
print(f"Is z/pi monotone increasing: {np.all(np.diff(z_synthetic / pik_desc) > 0)}")
print(f"z/pi (first 5): {(z_synthetic/pik_desc)[:5]}")

# verify با omega=0
omega_zero = np.zeros((n, N))
rho_zero   = np.zeros((n, N-1))

K_zero = np.real(CaDsd(pi=pik_desc, M=n, omega=omega_zero, rho=rho_zero)["K"])

V_ppi2 = Ppi(pik_desc)
K_ppi2 = np.real(V_ppi2 @ V_ppi2.T)

y_over_pi = z_synthetic / pik_desc

var_ppi  = float(y_over_pi @ (K_ppi2 * (np.eye(N) - K_ppi2)) @ y_over_pi)
var_zero = float(np.real(y_over_pi @ (K_zero * (np.eye(N) - np.conj(K_zero))) @ y_over_pi))

print(f"\nV_Ppi  = {var_ppi:.6f}")
print(f"V_zero = {var_zero:.6f}")
print(f"Are they equal? {np.isclose(var_ppi, var_zero, rtol=1e-3)}")

Is z/pi monotone increasing: True
z/pi (first 5): [49.4057971  50.16297245 50.59188392 50.97336508 51.6144169 ]

V_Ppi  = 709.867067
V_zero = 710.138555
Are they equal? True


In [17]:
# ============================================================
# WARM START DATASET — Vincent's approach (N=30, n=5)
# Pi descending, z/pi monotone increasing
# => omega=0 gives exactly Ppi variance (Rz,0 = 1)
# ============================================================
import numpy as np
import pandas as pd

np.random.seed(123)

N = 30
n = 5

# Step 1: Pi descending with small range (Vincent's suggestion)
Pi_raw = np.random.uniform(0.4, 0.8, N)
Pi_raw = Pi_raw * n / Pi_raw.sum()
Pi     = np.sort(Pi_raw)[::-1]   # descending

# Step 2: z such that z/pi is monotone increasing
z1_over_pi1 = 5.0 / Pi[0]
ratios = np.cumsum(np.concatenate([[z1_over_pi1],
                                    np.random.uniform(0.2, 1, N-1)]))
z = ratios * Pi

# Step 3: y arbitrary
y = z + np.random.normal(0, 0.1 * z.std(), N)

# Verify
assert np.all(np.diff(z / Pi) > 0), "z/pi not monotone!"
assert np.isclose(Pi.sum(), n), "sum(Pi) != n"
print(f"N={N}, n={n}")
print(f"Pi range: [{Pi.min():.4f}, {Pi.max():.4f}], sum={Pi.sum():.4f}")
print(f"z/pi monotone increasing: {np.all(np.diff(z/Pi) > 0)}")

# Verify omega=0 gives Ppi variance
omega_zero = np.zeros((n, N))
rho_zero   = np.zeros((n, N-1))
K_zero     = np.real(CaDsd(pi=Pi, M=n, omega=omega_zero, rho=rho_zero)["K"])
V_ppi_ws   = Ppi(Pi)
K_ppi_ws   = np.real(V_ppi_ws @ V_ppi_ws.T)
z_over_pi  = z / Pi
I_N        = np.eye(N)
var_ppi_ws  = float(z_over_pi @ (K_ppi_ws * (I_N - K_ppi_ws)) @ z_over_pi)
var_zero_ws = float(np.real(z_over_pi @ (K_zero * (I_N - np.conj(K_zero))) @ z_over_pi))
print(f"\nVerification — V_Ppi={var_ppi_ws:.4f}, V_zero={var_zero_ws:.4f}")
print(f"Equal? {np.isclose(var_ppi_ws, var_zero_ws, rtol=1e-3)}")

# SRS variances
var_srs_y = N**2 * (1 - n/N) * np.var(y, ddof=1) / n
var_srs_z = N**2 * (1 - n/N) * np.var(z, ddof=1) / n
print(f"\nvar_srs_y={var_srs_y:.2f}, var_srs_z={var_srs_z:.2f}")

N=30, n=5
Pi range: [0.1179, 0.2203], sum=5.0000
z/pi monotone increasing: True

Verification — V_Ppi=5.9480, V_zero=5.9504
Equal? True

var_srs_y=6.87, var_srs_z=6.98


In [15]:
# ============================================================
# ABC با warm start (omega=0)
# ============================================================

# ساخت ABC object
abc_ws = ABCAlgorithm(
    y_sorted    = y,
    z_sorted    = z,
    pik_sorted  = Pi,
    var_srs_y   = var_srs_y,
    var_srs_z   = var_srs_z,
    M           = n,
    n           = n,
    case_name   = "WARM_START_N30_n5",
    objective   = "eff_z",
    enforce_cadsd_order = True,
    random_state        = 12345,
    validation_mode     = "fast",
    eigen_check_interval     = 0,
    initial_strict_checks    = 2,
)

print(f"Ppi eff_z = {abc_ws.eff_z_optimal:.4f}")

# Warm start: inject omega=0 as first food source
omega_ws = np.zeros((n, N))
rho_ws   = np.random.default_rng(12345).random((n, N-1))

eff_z_ws, eff_y_ws, valid_ws = abc_ws.evaluate(omega_ws, rho_ws)
print(f"Warm start valid: {valid_ws}")
print(f"Warm start eff_z: {eff_z_ws:.4f}")
print(f"Warm start Rz,0 = {eff_z_ws / abc_ws.eff_z_optimal:.4f}")

# Override initialize_population to use omega=0 as first source
_original_init = abc_ws.initialize_population.__func__

def _warm_init(self, colony_size, verbose=True):
    if verbose:
        print(f"Initializing {colony_size} food sources (warm start)...")
    population = []

    # First source: omega=0 (warm start)
    food = self._food_from_arrays(omega_ws, rho_ws)
    if food is not None:
        population.append(food)
        if verbose:
            print(f"  Warm start injected: eff_z={food['eff_z']:.4f}")

    # Rest: random
    max_attempts = colony_size * 20
    count = 0
    while len(population) < colony_size and count < max_attempts:
        omega_r, rho_r = self._random_candidate()
        food = self._food_from_arrays(omega_r, rho_r)
        if food is not None:
            population.append(food)
        count += 1

    if verbose:
        print(f"Initialized {len(population)} food sources.")
        print(f"Best initial: eff_z={self.global_best_eff_z:.4f}")

    return population

import types
abc_ws.initialize_population = types.MethodType(_warm_init, abc_ws)

# Run ABC
results_ws = abc_ws.optimize(
    colony_size           = 20,
    max_iterations        = 1000,
    limit                 = 25,
    onlooker_factor       = 0.55,
    local_search_attempts = 1,
    local_search_interval = 12,
    progress_interval     = 50,
    early_stopping        = False,
)

Ppi eff_z = 1.1738
Warm start valid: True
Warm start eff_z: 1.1733
Warm start Rz,0 = 0.9996
ABC: WARM_START_N30_n5
Reference Pπ: eff_z=1.1738, eff_y=1.0949
ABC settings: colony=20, iter=1000, limit=25, progress_every=50
--------------------------------------------------------------------------------
Initializing 20 food sources (warm start)...
  Warm start injected: eff_z=1.1733
Initialized 20 food sources.
Best initial: eff_z=1.1733
  iter |      ABC_z |      ABC_y |   ABC/Pπ | scout |   valid/eval |  elapsed
-----------------------------------------------------------------------------
     1 |     1.1733 |     1.0944 |    1.000 |     0 |        52/52 |     2.5s
    50 |    10.4930 |     7.0828 |    8.939 |     0 |    1945/1950 |   111.0s
   100 |    43.8803 |    15.0698 |   37.384 |     4 |    4543/4554 |   242.0s
   150 |    43.8803 |    15.0698 |   37.384 |     0 |    6864/6883 |   379.9s
   200 |    43.8803 |    15.0698 |   37.384 |     2 |    9353/9387 |   526.8s
   250 |    43.8

In [25]:
# ============================================================
# ABC with warm start (omega=0) + Random Search baseline
# ============================================================

abc_ws = ABCAlgorithm(
    y_sorted    = y,
    z_sorted    = z,
    pik_sorted  = Pi,
    var_srs_y   = var_srs_y,
    var_srs_z   = var_srs_z,
    M           = n,
    n           = n,
    case_name   = "WARM_START_N30_n5",
    objective   = "eff_z",
    enforce_cadsd_order  = True,
    random_state         = 12345,
    validation_mode      = "fast",
    eigen_check_interval = 0,
    initial_strict_checks= 2,
)

random_ws = RandomSearchAlgorithm(
    y_sorted    = y,
    z_sorted    = z,
    pik_sorted  = Pi,
    var_srs_y   = var_srs_y,
    var_srs_z   = var_srs_z,
    M           = n,
    n           = n,
    case_name   = "WARM_START_RANDOM_N30_n5",
    objective   = "eff_z",
    enforce_cadsd_order  = True,
    random_state         = 54321,
    validation_mode      = "fast",
    eigen_check_interval = 0,
    initial_strict_checks= 2,
)

print(f"Ppi eff_z = {abc_ws.eff_z_optimal:.4f}")

# Warm start: omega=0
omega_ws = np.zeros((n, N))
rho_ws   = np.random.default_rng(12345).random((n, N-1))

eff_z_ws, eff_y_ws, valid_ws = abc_ws.evaluate(omega_ws, rho_ws)
print(f"Warm start valid: {valid_ws}")
print(f"Warm start Rz,0 = {eff_z_ws / abc_ws.eff_z_optimal:.4f}")

# Override initialize_population
import types

def _warm_init(self, colony_size, verbose=True):
    if verbose:
        print(f"Initializing {colony_size} food sources (warm start)...")
    population = []

    food = self._food_from_arrays(omega_ws, rho_ws)
    if food is not None:
        population.append(food)
        if verbose:
            print(f"  Warm start injected: eff_z={food['eff_z']:.4f}")

    max_attempts = colony_size * 20
    count = 0
    while len(population) < colony_size and count < max_attempts:
        omega_r, rho_r = self._random_candidate()
        food = self._food_from_arrays(omega_r, rho_r)
        if food is not None:
            population.append(food)
        count += 1

    if verbose:
        print(f"Initialized {len(population)} food sources.")
        print(f"Best initial: eff_z={self.global_best_eff_z:.4f}, eff_y={self.global_best_eff_y:.4f}")

    return population

abc_ws.initialize_population = types.MethodType(_warm_init, abc_ws)

# Run ABC + Random Search
results_ws = abc_ws.optimize(
    colony_size           = 20,
    max_iterations        = 1000,
    limit                 = 25,
    onlooker_factor       = 0.55,
    local_search_attempts = 1,
    local_search_interval = 12,
    progress_interval     = 100,
    early_stopping        = False,
    random_searcher       = random_ws,
    random_start_evals    = 20,
    random_evals_per_iteration = 31,
)

Ppi eff_z = 1.1738
Warm start valid: True
Warm start Rz,0 = 0.9996
 ABC ALGORITHM - WARM_START_N30_n5
 Configuration:
   colony_size          = 20
   max_iterations       = 1000
   abandonment limit    = 25
   objective            = eff_z
   validation mode      = fast
   onlooker factor      = 0.55
   early stopping       = False, patience=250
   time limit           = None
   random search        = ON
   random start evals   = 20
   random evals/iter    = 31
   progress interval    = every 100 iterations
   local search interval= every 12 iterations
 Reference Pπ efficiency versus SRS:
   Pπ_z = 1.17
   Pπ_y = 1.09

Initializing 20 food sources (warm start)...
  Warm start injected: eff_z=1.1733
Initialized 20 food sources.
Best initial: eff_z=1.1733, eff_y=1.0944
 iter | ABC_z/Pπ_z | ABC_y/Pπ_y | Rand_z/Pπ_z | Rand_y/Pπ_y | scout | ABC val% | Rnd val% |  elapsed
----------------------------------------------------------------------------------------------------
    1 |       1.00 | 

In [26]:
print(f"Corr(y, z) = {np.corrcoef(y, z)[0,1]:.4f}")

Corr(y, z) = 0.9959


# Final Run

In [11]:
import pandas as pd
df_syn = pd.read_csv(
    r'C:\Users\Amir\Desktop\GFS\graphical-sampling\simulations_abc\store\synthetic_population_N30_y_z_pi.csv'
)
print(df_syn.shape)
print(df_syn.columns.tolist())
print(df_syn.head())

(30, 8)
['id', 'y', 'z70', 'z80', 'z90', 'x75', 'x90', 'equal']
   id           y         z70         z80         z90         x75         x90  \
0   1  107.802154   93.187947  110.478533  116.710761  130.979858  122.907026   
1   2  110.533598   77.745743  127.236432  100.183080  100.974223  100.106119   
2   3   89.514778   99.774190  100.401508   97.594247   67.260948   78.608363   
3   4  100.438333  109.419545   88.757808   92.251437   91.210994   94.711583   
4   5  102.022818   95.828759   94.579228  111.557499  124.238351   88.639669   

   equal  
0    1.0  
1    1.0  
2    1.0  
3    1.0  
4    1.0  


In [12]:
# ============================================================
# WARM START DATASET — Vincent's approach
# ============================================================
import pandas as pd
import numpy as np

df_pop = pd.read_csv(
    r'C:\Users\Amir\Desktop\GFS\graphical-sampling\simulations_abc\store\synthetic_population_N30_y_z_pi.csv'
)

N = 30
n = 5

y_raw  = df_pop['y'].values
z_raw  = df_pop['z90'].values    # z90: corr=0.90 با y
x_raw  = df_pop['x75'].values    # x75: برای Pi

print(f"Corr(y, z90) = {np.corrcoef(y_raw, z_raw)[0,1]:.3f}")
print(f"Corr(y, x75) = {np.corrcoef(y_raw, x_raw)[0,1]:.3f}")

# ساخت Pi از x75
pik_raw = inclusionprobabilities(x_raw, n)
print(f"Pi sum = {pik_raw.sum():.4f}")
print(f"Pi range: [{pik_raw.min():.4f}, {pik_raw.max():.4f}]")

# Pi descending مرتب کن
desc_idx = np.argsort(pik_raw)[::-1]
Pi       = pik_raw[desc_idx]
z        = z_raw[desc_idx]
y        = y_raw[desc_idx]

# چک شرط Vincent: z/pi monotone؟
print(f"\nIs z/pi monotone increasing: {np.all(np.diff(z/Pi) > 0)}")
print(f"Is z/pi monotone decreasing: {np.all(np.diff(z/Pi) < 0)}")

Corr(y, z90) = 0.900
Corr(y, x75) = 0.750
Pi sum = 5.0000
Pi range: [0.0945, 0.2248]

Is z/pi monotone increasing: False
Is z/pi monotone decreasing: False


In [16]:
# چک رابطه Pi و y چیه
print(f"Corr(Pi, y) = {np.corrcoef(Pi, y_desc)[0,1]:.4f}")
print(f"Corr(Pi, y/Pi) = {np.corrcoef(Pi, y_desc/Pi)[0,1]:.4f}")


z_test = np.sort(y_desc/Pi) * Pi  # z/pi = sorted y/pi (increasing)
print(f"\nCorr(y, z_test) = {np.corrcoef(y_desc, z_test)[0,1]:.4f}")
print(f"Monotone: {np.all(np.diff(z_test/Pi) > 0)}")

Corr(Pi, y) = 0.7500
Corr(Pi, y/Pi) = -0.4417

Corr(y, z_test) = 0.6047
Monotone: True


# Sensitivity Analysis (moved to the end)


In [85]:
import numpy as np
import pandas as pd


def _abc_variance_from_omega_rho(abc: ABCAlgorithm, omega: np.ndarray, rho: np.ndarray):
    """Return kernel, variances, efficiencies, and validation details for a candidate."""
    try:
        K_dict = CaDsd(pi=abc.pik_sorted, M=abc.M, omega=omega, rho=rho)
        Kmat = K_dict["K"].astype(np.complex128)
    except Exception as e:
        return None, {
            "valid": False,
            "reason": f"CaDsd error: {str(e)[:80]}",
        }

    diag_K = np.real(np.diag(Kmat))
    max_diag_diff = float(np.max(np.abs(diag_K - abc.pik_sorted_desc)))

    if not np.allclose(diag_K, abc.pik_sorted_desc, atol=1e-3):
        return Kmat, {
            "valid": False,
            "reason": f"diagonal mismatch, max diff={max_diag_diff:.3e}",
        }

    try:
        evals = np.linalg.eigvalsh(Kmat)
    except Exception as e:
        return Kmat, {
            "valid": False,
            "reason": f"eigvalsh error: {str(e)[:80]}",
        }

    eig_min = float(evals.min())
    eig_max = float(evals.max())
    trace = float(evals.sum())

    if not (np.all(evals >= -1e-3) and np.all(evals <= 1 + 1e-3)):
        return Kmat, {
            "valid": False,
            "reason": f"eigenvalues outside [0,1], min={eig_min:.3e}, max={eig_max:.3e}",
            "eig_min": eig_min,
            "eig_max": eig_max,
            "trace": trace,
            "max_diag_diff": max_diag_diff,
        }

    if not np.isclose(trace, abc.n, atol=1e-3):
        return Kmat, {
            "valid": False,
            "reason": f"trace mismatch, trace={trace:.6f}, n={abc.n}",
            "eig_min": eig_min,
            "eig_max": eig_max,
            "trace": trace,
            "max_diag_diff": max_diag_diff,
        }

    A = Kmat * (abc.I_N - Kmat.conj())
    var_y = float(np.real(abc.y_sorted.conj().T @ abc.Dpi_inv @ A @ abc.Dpi_inv @ abc.y_sorted))
    var_z = float(np.real(abc.z_sorted.conj().T @ abc.Dpi_inv @ A @ abc.Dpi_inv @ abc.z_sorted))

    eff_y = abc.var_srs_y / var_y if var_y > 0 else np.nan
    eff_z = abc.var_srs_z / var_z if var_z > 0 else np.nan

    return Kmat, {
        "valid": True,
        "reason": "OK",
        "var_y": var_y,
        "var_z": var_z,
        "eff_y": eff_y,
        "eff_z": eff_z,
        "eig_min": eig_min,
        "eig_max": eig_max,
        "trace": trace,
        "max_diag_diff": max_diag_diff,
    }


def sensitivity_analysis_uniform(
    abc: ABCAlgorithm,
    result: dict,
    epsilons=(0.01, 0.02),
    case_name: Optional[str] = None,
    print_table: bool = True,
) -> pd.DataFrame:
    """
    Concise uniform sensitivity analysis around the best ABC design.

    It tests eight perturbation directions for each epsilon:
    omega/rho +/-, omega only +/-, and rho only +/-.
    """
    if result.get("best_omega") is None or result.get("best_rho") is None:
        raise ValueError("The result does not contain best_omega/best_rho. Run ABC first.")

    base_omega = result["best_omega"]
    base_rho = result["best_rho"]
    if case_name is None:
        case_name = result.get("case", abc.case_name)

    _, base_info = _abc_variance_from_omega_rho(abc, base_omega, base_rho)
    if not base_info["valid"]:
        raise ValueError(f"Base design is not valid: {base_info['reason']}")

    perturbation_cases = [
        ("omega+, rho+", +1, +1),
        ("omega+, rho-", +1, -1),
        ("omega-, rho+", -1, +1),
        ("omega-, rho-", -1, -1),
        ("omega+, rho=0", +1, 0),
        ("omega-, rho=0", -1, 0),
        ("omega=0, rho+", 0, +1),
        ("omega=0, rho-", 0, -1),
    ]

    rows = []
    print("\n" + "=" * 90)
    print(f"SENSITIVITY ANALYSIS: {case_name}")
    print("=" * 90)
    print(
        f"Base: var_y={base_info['var_y']:.6g}, var_z={base_info['var_z']:.6g}, "
        f"eff_y={base_info['eff_y']:.4f}, eff_z={base_info['eff_z']:.4f}"
    )

    for eps in epsilons:
        print(f"\nEpsilon = {eps}")
        print("-" * 90)
        print(f"{'case':<18} {'valid':>6} {'d_var_y%':>10} {'d_var_z%':>10} {'d_eff_z%':>10}  reason")
        print("-" * 90)

        for label, s_omega, s_rho in perturbation_cases:
            omega_new = np.clip(base_omega + s_omega * eps, 0.0, 1.0)
            rho_new = np.clip(base_rho + s_rho * eps, 0.0, 1.0)

            _, info = _abc_variance_from_omega_rho(abc, omega_new, rho_new)

            row = {
                "case": case_name,
                "epsilon": eps,
                "perturbation_type": label,
                "valid": info["valid"],
                "reason": info["reason"],
                "base_var_y": base_info["var_y"],
                "base_var_z": base_info["var_z"],
                "base_eff_y": base_info["eff_y"],
                "base_eff_z": base_info["eff_z"],
            }

            if info["valid"]:
                row.update({
                    "var_y": info["var_y"],
                    "var_z": info["var_z"],
                    "eff_y": info["eff_y"],
                    "eff_z": info["eff_z"],
                    "delta_var_y_percent": 100.0 * (info["var_y"] / base_info["var_y"] - 1.0),
                    "delta_var_z_percent": 100.0 * (info["var_z"] / base_info["var_z"] - 1.0),
                    "delta_eff_z_percent": 100.0 * (info["eff_z"] / base_info["eff_z"] - 1.0),
                    "eig_min": info["eig_min"],
                    "eig_max": info["eig_max"],
                    "trace": info["trace"],
                    "max_diag_diff": info["max_diag_diff"],
                })
                print(
                    f"{label:<18} {str(True):>6} "
                    f"{row['delta_var_y_percent']:>+10.3f} "
                    f"{row['delta_var_z_percent']:>+10.3f} "
                    f"{row['delta_eff_z_percent']:>+10.3f}  OK"
                )
            else:
                row.update({
                    "var_y": np.nan,
                    "var_z": np.nan,
                    "eff_y": np.nan,
                    "eff_z": np.nan,
                    "delta_var_y_percent": np.nan,
                    "delta_var_z_percent": np.nan,
                    "delta_eff_z_percent": np.nan,
                })
                print(f"{label:<18} {str(False):>6} {'nan':>10} {'nan':>10} {'nan':>10}  {info['reason']}")

            rows.append(row)

    df_sens = pd.DataFrame(rows)

    if print_table:
        valid_df = df_sens[df_sens["valid"] == True]
        print("\n" + "=" * 90)
        print("SENSITIVITY SUMMARY")
        print("=" * 90)
        print(f"Valid perturbations: {len(valid_df)}/{len(df_sens)}")
        if len(valid_df) > 0:
            print(
                f"Mean delta var_z: {valid_df['delta_var_z_percent'].mean():+.3f}% "
                f"(min {valid_df['delta_var_z_percent'].min():+.3f}%, "
                f"max {valid_df['delta_var_z_percent'].max():+.3f}%)"
            )
            print(
                f"Mean delta eff_z: {valid_df['delta_eff_z_percent'].mean():+.3f}% "
                f"(min {valid_df['delta_eff_z_percent'].min():+.3f}%, "
                f"max {valid_df['delta_eff_z_percent'].max():+.3f}%)"
            )

    return df_sens


print("Sensitivity functions loaded.")

# Run sensitivity after all ABC runs.
# By default this analyzes only the last completed ABC run, so the output stays concise.
# To analyze another run, set SENSITIVITY_KEY to one key from abc_run_details.keys().

RUN_SENSITIVITY = True
SENSITIVITY_KEY = None      # None -> use the last run
SENSITIVITY_EPSILONS = (0.01, 0.02)

if RUN_SENSITIVITY:
    if "abc_run_details" not in globals() or len(abc_run_details) == 0:
        print("No ABC runs found yet. Run the ABC section first, then run this cell.")
    else:
        if SENSITIVITY_KEY is None:
            SENSITIVITY_KEY = list(abc_run_details.keys())[-1]

        print(f"Available run keys: {list(abc_run_details.keys())}")
        print(f"Running sensitivity for: {SENSITIVITY_KEY}")

        selected = abc_run_details[SENSITIVITY_KEY]
        df_sensitivity = sensitivity_analysis_uniform(
            abc=selected["abc"],
            result=selected["result"],
            epsilons=SENSITIVITY_EPSILONS,
            case_name=SENSITIVITY_KEY,
            print_table=True,
        )

        df_sensitivity.to_csv(f"sensitivity_{SENSITIVITY_KEY}.csv", index=False)
        print(f"\nSaved: sensitivity_{SENSITIVITY_KEY}.csv")
else:
    print("Sensitivity is skipped. Set RUN_SENSITIVITY = True when you want to run it.")



Sensitivity functions loaded.
Available run keys: ['z70_x75_N30_n5', 'z80_x75_N30_n5', 'z90_x75_N30_n5', 'z70_x90_N30_n5', 'z80_x90_N30_n5', 'z90_x90_N30_n5', 'z70_equal_N30_n5', 'z80_equal_N30_n5', 'z90_equal_N30_n5']
Running sensitivity for: z90_equal_N30_n5


ValueError: Base design is not valid: eigenvalues outside [0,1], min=-3.925e-07, max=1.009e+00